# 17 — Geological CO₂ Storage Integration into GeoCANOE

## Purpose

This notebook explores integration of the **CanCO₂ Storage Unified Database** into the GeoCANOE spatial modelling workflow.

The unified GeoPackage is treated here as an external **Silver-layer geological storage input**. The immediate objective is to determine how heterogeneous geological storage geometries can be mapped onto the existing GeoCANOE regional basemap and subsequently represented within the Temoa/CANOE optimization structure.

This notebook is intentionally exploratory. The spatial logic developed here will be validated before being generalized and migrated into the GeoCANOE `src/` workflow.

---

## Initial scope

The first implementation will use the **25 km projected basemap**.

This resolution is useful for developing the integration logic because the geological storage database contains heterogeneous spatial representations that do not necessarily correspond directly to the regular GeoCANOE regional polygons. The workflow therefore needs an explicit mapping between:

\[
\text{storage geometry}
\rightarrow
\text{spatial overlap}
\rightarrow
\text{GeoCANOE region}
\rightarrow
\text{optimization representation}
\]

The 25 km case will be used to develop and inspect this mapping before generalizing the procedure across GeoCANOE basemap resolutions.

---

## Phase 1 — Binary storage accessibility

The initial model representation will deliberately ignore quantitative storage capacity and injection cost.

For each GeoCANOE region, the spatial overlay will determine whether geological storage potential intersects the region. Qualifying regions will initially be treated as having effectively unconstrained geological storage availability.

Conceptually:

\[
A_r =
\begin{cases}
1, & \text{if qualifying storage geometry overlaps region } r \\
0, & \text{otherwise}
\end{cases}
\]

where \(A_r\) represents regional geological-storage accessibility.

This provides a minimal first representation analogous to the earlier spatial mapping of transportation infrastructure: establish **where the technology or resource is spatially available before introducing detailed techno-economic constraints**.

The resulting regional storage-accessibility layer can then be connected to the Temoa/CANOE representation of CO₂ transport and geological injection.

---

## Important semantic distinction

The unified database contains several different concepts that must remain distinct during spatial aggregation:

- geological storage units;
- spatial storage representations;
- geological assessments;
- regulatory or administrative features.

Spatial overlap alone therefore does **not** imply demonstrated storage capacity, injectivity, permitting, legal storage rights, or project-level suitability.

The first-pass binary representation should consequently be interpreted only as:

> **A GeoCANOE region spatially intersects geological information that qualifies it as a candidate region for geological CO₂ storage under the selected configuration.**

It should not be interpreted as a quantitative storage resource.

---

## Development sequence

This notebook will proceed through the following conceptual stages:

1. Inspect the unified storage GeoPackage and identify the spatial layers and attributes required for model integration.
2. Load the GeoCANOE 25 km projected basemap.
3. Harmonize coordinate reference systems and geometry assumptions.
4. Examine the different storage geometry types and their implications for regional aggregation.
5. Develop and validate storage-feature → GeoCANOE-region spatial mapping.
6. Construct an initial binary regional storage-accessibility representation.
7. Explore how storage availability should enter the Temoa/CANOE formulation.
8. Separate spatial preprocessing logic from optimization constraints in preparation for migration into `src/`.

---

## Future extensions

The binary representation is an intermediate modelling assumption.

Once additional geological information is available, the storage representation can be progressively extended to distinguish:

\[
\text{spatial overlap}
\rightarrow
\text{candidate storage}
\rightarrow
\text{estimated capacity}
\rightarrow
\text{injectivity}
\rightarrow
\text{injection cost}
\]

Injection-cost information will provide the next refinement, while quantitative storage-capacity constraints can be introduced later where scientifically defensible data are available.

The eventual implementation should therefore support configurable storage representations rather than embedding a single interpretation of geological overlap directly into the optimization model.

---

## Architecture note

For this notebook, the unified GeoPackage will be read directly from the local `canco2-storage` processed-data output.

The longer-term dependency between `canco2-storage` and `temoa_geospace` / GeoCANOE is intentionally left unresolved. Potential mechanisms include cached artifacts, packaged releases, or a Python package interface. Dependency management will be addressed only after the required integration API is understood from this prototype.

In [1]:
# ---------------------------------------------------------------------------
# Cell 2 — Define unified geological storage input and inspect GeoPackage
# ---------------------------------------------------------------------------

import re
import sys
from pathlib import Path

import pandas as pd
import sqlite3

import geopandas as gpd
import pyarrow as pa

import pyogrio
import shapely
from shapely.ops import unary_union
import lonboard
from lonboard import Map, PolygonLayer


# ---------------------------------------------------------------------------
# Input path
# ---------------------------------------------------------------------------

STORAGE_GPKG = Path(
    r"C:\Users\aviga\Research\repos\canco2-storage"
    r"\data\processed\unified_storage"
    r"\20260917_13_CanadaGeologicalStorageUnified_AV_v2.gpkg"
)

if not STORAGE_GPKG.exists():
    raise FileNotFoundError(
        f"Unified geological storage GeoPackage not found:\n{STORAGE_GPKG}"
    )


# ---------------------------------------------------------------------------
# Inspect registered GeoPackage layers
# ---------------------------------------------------------------------------

layers = pyogrio.list_layers(STORAGE_GPKG)

print(f"GeoPackage: {STORAGE_GPKG.name}")
print(f"Number of registered layers/tables: {len(layers)}")
print()

for layer_name, geometry_type in layers:
    geometry_label = geometry_type if geometry_type else "non-spatial"
    print(f"{layer_name:<55} {geometry_label}")

GeoPackage: 20260917_13_CanadaGeologicalStorageUnified_AV_v2.gpkg
Number of registered layers/tables: 9

storage_features                                        Unknown
administrative_features                                 MultiPolygon
storage_units                                           non-spatial
storage_assessments                                     non-spatial
source_catalog_canada_geological_storage_unified        non-spatial
source_metadata_canada_geological_storage_unified       non-spatial
source_qa_canada_geological_storage_unified             non-spatial
metadata_canada_geological_storage_unified              non-spatial
qa_canada_geological_storage_unified                    non-spatial


In [2]:
# ---------------------------------------------------------------------------
# Cell 3 — Load canonical geological storage tables
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Spatial storage representations
# ---------------------------------------------------------------------------

storage_features = gpd.read_file(
    STORAGE_GPKG,
    layer="storage_features",
)

print("STORAGE FEATURES")
print("-" * 80)
print(f"Rows:          {len(storage_features):,}")
print(f"CRS:           {storage_features.crs}")
print(f"Geometry name: {storage_features.geometry.name}")
print()

print("Geometry types:")
print(storage_features.geometry.geom_type.value_counts(dropna=False))
print()

print("Columns:")
print(storage_features.columns.tolist())


# ---------------------------------------------------------------------------
# Non-spatial canonical tables
# ---------------------------------------------------------------------------

with sqlite3.connect(STORAGE_GPKG) as conn:
    storage_units = pd.read_sql_query(
        "SELECT * FROM storage_units",
        conn,
    )

    storage_assessments = pd.read_sql_query(
        "SELECT * FROM storage_assessments",
        conn,
    )


# ---------------------------------------------------------------------------
# Basic table structure
# ---------------------------------------------------------------------------

print("\nSTORAGE UNITS")
print("-" * 80)
print(f"Rows: {len(storage_units):,}")
print("Columns:")
print(storage_units.columns.tolist())

print("\nSTORAGE ASSESSMENTS")
print("-" * 80)
print(f"Rows: {len(storage_assessments):,}")
print("Columns:")
print(storage_assessments.columns.tolist())

STORAGE FEATURES
--------------------------------------------------------------------------------
Rows:          35,320
CRS:           EPSG:3978
Geometry name: geometry

Geometry types:
MultiPolygon          35295
Polygon                  24
GeometryCollection        1
Name: count, dtype: int64

Columns:
['storage_feature_id', 'storage_unit_id', 'source_dataset', 'source_layer', 'source_feature_id', 'storage_type', 'storage_subtype', 'representation', 'assessment_type', 'data_class', 'capacity_data', 'injectivity_status', 'country', 'province_territory', 'land_status', 'geometry_area_m2', 'geometry_area_ha', 'geometry_perimeter_m', 'geometry']

STORAGE UNITS
--------------------------------------------------------------------------------
Rows: 2,843
Columns:
['fid', 'storage_unit_id', 'source_dataset', 'source_unit_id', 'storage_type', 'storage_subtype', 'storage_name', 'formation', 'geological_group', 'basin_name', 'country', 'province_territory', 'land_status', 'assessment_type', 'da

In [3]:
# ---------------------------------------------------------------------------
# Cell 4 — Characterize storage feature semantics
# ---------------------------------------------------------------------------

def summarize_column(gdf, column):
    """Print value counts for a categorical storage-feature field."""
    print(f"\n{column.upper()}")
    print("-" * 80)
    print(gdf[column].value_counts(dropna=False))


# ---------------------------------------------------------------------------
# Core semantic fields
# ---------------------------------------------------------------------------

summary_fields = [
    "source_dataset",
    "source_layer",
    "storage_type",
    "storage_subtype",
    "representation",
    "assessment_type",
    "data_class",
    "capacity_data",
    "injectivity_status",
    "province_territory",
    "land_status",
]

for field in summary_fields:
    summarize_column(storage_features, field)


# ---------------------------------------------------------------------------
# Geometry type by source dataset
# ---------------------------------------------------------------------------

geometry_by_source = pd.crosstab(
    storage_features["source_dataset"],
    storage_features.geometry.geom_type,
    margins=True,
)

print("\nGEOMETRY TYPE BY SOURCE DATASET")
print("-" * 80)
display(geometry_by_source)


# ---------------------------------------------------------------------------
# Representation by source dataset
# ---------------------------------------------------------------------------

representation_by_source = pd.crosstab(
    storage_features["source_dataset"],
    storage_features["representation"],
    margins=True,
)

print("\nREPRESENTATION BY SOURCE DATASET")
print("-" * 80)
display(representation_by_source)


# ---------------------------------------------------------------------------
# Storage type by source dataset
# ---------------------------------------------------------------------------

storage_type_by_source = pd.crosstab(
    storage_features["source_dataset"],
    storage_features["storage_type"],
    margins=True,
)

print("\nSTORAGE TYPE BY SOURCE DATASET")
print("-" * 80)
display(storage_type_by_source)

# ---------------------------------------------------------------------------
# Assessment scope and linkage structure
# ---------------------------------------------------------------------------

print("\nASSESSMENT SCOPE")
print("-" * 80)

print(
    storage_assessments["assessment_scope"]
    .value_counts(dropna=False)
)

# ---------------------------------------------------------------------------
# Assessment scope by source dataset
# ---------------------------------------------------------------------------

assessment_scope_by_source = pd.crosstab(
    storage_assessments["source_dataset"],
    storage_assessments["assessment_scope"],
    margins=True,
)

print("\nASSESSMENT SCOPE BY SOURCE DATASET")
print("-" * 80)
display(assessment_scope_by_source)

# ---------------------------------------------------------------------------
# Assessment linkage by source dataset
# ---------------------------------------------------------------------------

assessment_linkage_by_source = (
    storage_assessments
    .groupby("source_dataset", dropna=False)
    .agg(
        assessments=("storage_assessment_id", "size"),
        feature_linked=(
            "storage_feature_id",
            lambda x: x.notna().sum(),
        ),
        unit_linked=(
            "storage_unit_id",
            lambda x: x.notna().sum(),
        ),
    )
)

print("\nASSESSMENT LINKAGE BY SOURCE DATASET")
print("-" * 80)
display(assessment_linkage_by_source)

# ---------------------------------------------------------------------------
# Capacity availability by source dataset
# ---------------------------------------------------------------------------

capacity_availability_by_source = (
    storage_assessments
    .groupby("source_dataset", dropna=False)
    .agg(
        assessments=("storage_assessment_id", "size"),
        p10_available=(
            "storage_p10_tonnes",
            lambda x: x.notna().sum(),
        ),
        p50_available=(
            "storage_p50_tonnes",
            lambda x: x.notna().sum(),
        ),
        p90_available=(
            "storage_p90_tonnes",
            lambda x: x.notna().sum(),
        ),
        theoretical_available=(
            "theoretical_storage_tonnes",
            lambda x: x.notna().sum(),
        ),
        effective_available=(
            "effective_storage_tonnes",
            lambda x: x.notna().sum(),
        ),
    )
)

print("\nCAPACITY AVAILABILITY BY SOURCE DATASET")
print("-" * 80)
display(capacity_availability_by_source)


SOURCE_DATASET
--------------------------------------------------------------------------------
source_dataset
NATCARB             29271
ATLANTIC_COS         4711
BC_STORAGE_ATLAS     1338
Name: count, dtype: int64

SOURCE_LAYER
--------------------------------------------------------------------------------
source_layer
saline_resource_cells    26694
storage_units             4711
oil_gas_resources         1358
pool_features             1309
coal_resource_cells       1219
aquifer_features            29
Name: count, dtype: int64

STORAGE_TYPE
--------------------------------------------------------------------------------
storage_type
saline_aquifer                    26723
None                               4711
depleted_hydrocarbon_reservoir     2667
coal                               1219
Name: count, dtype: int64

STORAGE_SUBTYPE
--------------------------------------------------------------------------------
storage_subtype
None    35320
Name: count, dtype: int64

REPRESENTATION


col_0,GeometryCollection,MultiPolygon,Polygon,All
source_dataset,,,,
ATLANTIC_COS,0,4711,0,4711
BC_STORAGE_ATLAS,1,1313,24,1338
NATCARB,0,29271,0,29271
All,1,35295,24,35320



REPRESENTATION BY SOURCE DATASET
--------------------------------------------------------------------------------


representation,aquifer_extent,pool_extent,prospectivity_polygon,resource_grid_cell,storage_resource,All
source_dataset,,,,,,
ATLANTIC_COS,0,0,4711,0,0,4711
BC_STORAGE_ATLAS,29,1309,0,0,0,1338
NATCARB,0,0,0,27913,1358,29271
All,29,1309,4711,27913,1358,35320



STORAGE TYPE BY SOURCE DATASET
--------------------------------------------------------------------------------


...

assessment_scope,feature,unit,All
source_dataset,,,
ATLANTIC_COS,4711,0,4711
BC_STORAGE_ATLAS,0,1263,1263
NATCARB,29271,0,29271
All,33982,1263,35245



ASSESSMENT LINKAGE BY SOURCE DATASET
--------------------------------------------------------------------------------


,assessments,feature_linked,unit_linked
source_dataset,,,
ATLANTIC_COS,4711,4711,4711
BC_STORAGE_ATLAS,1263,0,1263
NATCARB,29271,29271,29271



CAPACITY AVAILABILITY BY SOURCE DATASET
--------------------------------------------------------------------------------


,assessments,p10_available,p50_available,p90_available,theoretical_available,effective_available
source_dataset,,,,,,
ATLANTIC_COS,4711,0,0,0,0,0
BC_STORAGE_ATLAS,1263,25,25,25,1263,1238
NATCARB,29271,23024,23024,23024,0,0


In [4]:
# ---------------------------------------------------------------------------
# Cell 5 — Load GeoCANOE 25 km projected basemap
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Basemap input
# ---------------------------------------------------------------------------

BASEMAP_GPKG = Path(
    r"C:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace"
    r"\data_files\processed\basemaps"
    r"\provinces_only_basemap_25km_centroid.gpkg"
)

print(BASEMAP_GPKG)
print("Exists:", BASEMAP_GPKG.exists())


# ---------------------------------------------------------------------------
# Inspect available layers
# ---------------------------------------------------------------------------

basemap_layers = pyogrio.list_layers(BASEMAP_GPKG)

print(f"GeoPackage: {BASEMAP_GPKG.name}")
print(f"Number of registered layers/tables: {len(basemap_layers)}")
print()

for layer_name, geometry_type in basemap_layers:
    geometry_label = geometry_type if geometry_type else "non-spatial"
    print(f"{layer_name:<55} {geometry_label}")

C:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\basemaps\provinces_only_basemap_25km_centroid.gpkg
Exists: True
GeoPackage: provinces_only_basemap_25km_centroid.gpkg
Number of registered layers/tables: 1

regions                                                 Polygon


In [5]:
# ---------------------------------------------------------------------------
# Cell 6 — Load and inspect GeoCANOE 25 km regions
# ---------------------------------------------------------------------------

regions_25km = gpd.read_file(
    BASEMAP_GPKG,
    layer="regions",
)


# ---------------------------------------------------------------------------
# Basic structure
# ---------------------------------------------------------------------------

print("GEOCANOE 25 KM REGIONS")
print("-" * 80)

print(f"Rows:          {len(regions_25km):,}")
print(f"CRS:           {regions_25km.crs}")
print(f"Geometry name: {regions_25km.geometry.name}")
print()

print("Geometry types:")
print(regions_25km.geometry.geom_type.value_counts(dropna=False))
print()

print("Columns:")
print(regions_25km.columns.tolist())


# ---------------------------------------------------------------------------
# Geometry QA
# ---------------------------------------------------------------------------

print("\nGEOMETRY QA")
print("-" * 80)

print(f"Null geometries:    {regions_25km.geometry.isna().sum():,}")
print(f"Empty geometries:   {regions_25km.geometry.is_empty.sum():,}")
print(f"Invalid geometries: {(~regions_25km.geometry.is_valid).sum():,}")


# ---------------------------------------------------------------------------
# Preview
# ---------------------------------------------------------------------------

display(regions_25km.head())

GEOCANOE 25 KM REGIONS
--------------------------------------------------------------------------------
Rows:          9,269
CRS:           EPSG:3347
Geometry name: geometry

Geometry types:
Polygon    9269
Name: count, dtype: int64

Columns:
['region', 'site_id', 'study_area', 'province_codes', 'grid_type', 'grid_crs', 'resolution', 'resolution_unit', 'resolution_deg', 'resolution_km', 'cell_size_native', 'keep_method', 'x_min', 'x_max', 'y_min', 'y_max', 'centroid_x', 'centroid_y', 'lon', 'lat', 'geometry']

GEOMETRY QA
--------------------------------------------------------------------------------
Null geometries:    0
Empty geometries:   0
Invalid geometries: 0


,region,site_id,study_area,province_codes,grid_type,grid_crs,resolution,resolution_unit,resolution_deg,resolution_km,...,keep_method,x_min,x_max,y_min,y_max,centroid_x,centroid_y,lon,lat,geometry
0,R0,R0,provinces_only,"NL,PE,NS,NB,QC,ON,MB,SK,AB,BC",projected,EPSG:3347,25.0,km,NaN,25.0,...,centroid,6950000.0,6975000.0,700000.0,725000.0,6962500.0,712500.0,-82.896136,42.172075,"POLYGON ((6975000 700000, 6975000 725000, 6950..."
1,R1,R1,provinces_only,"NL,PE,NS,NB,QC,ON,MB,SK,AB,BC",projected,EPSG:3347,25.0,km,NaN,25.0,...,centroid,6975000.0,7000000.0,700000.0,725000.0,6987500.0,712500.0,-82.606095,42.140933,"POLYGON ((7000000 700000, 7000000 725000, 6975..."
2,R2,R2,provinces_only,"NL,PE,NS,NB,QC,ON,MB,SK,AB,BC",projected,EPSG:3347,25.0,km,NaN,25.0,...,centroid,7000000.0,7025000.0,725000.0,750000.0,7012500.0,737500.0,-82.272450,42.324460,"POLYGON ((7025000 725000, 7025000 750000, 7000..."
3,R3,R3,provinces_only,"NL,PE,NS,NB,QC,ON,MB,SK,AB,BC",projected,EPSG:3347,25.0,km,NaN,25.0,...,centroid,7025000.0,7050000.0,725000.0,750000.0,7037500.0,737500.0,-81.981905,42.291177,"POLYGON ((7050000 725000, 7050000 750000, 7025..."
4,R4,R4,provinces_only,"NL,PE,NS,NB,QC,ON,MB,SK,AB,BC",projected,EPSG:3347,25.0,km,NaN,25.0,...,centroid,6975000.0,7000000.0,750000.0,775000.0,6987500.0,762500.0,-82.520312,42.572812,"POLYGON ((7000000 750000, 7000000 775000, 6975..."


In [6]:
# ---------------------------------------------------------------------------
# Cell 7 — Align geological storage features to GeoCANOE basemap CRS
# ---------------------------------------------------------------------------

# The target CRS is determined by the selected GeoCANOE basemap.
target_crs = regions_25km.crs

if target_crs is None:
    raise ValueError("GeoCANOE basemap does not define a CRS.")

if storage_features.crs is None:
    raise ValueError("Geological storage features do not define a CRS.")


# ---------------------------------------------------------------------------
# Report CRS transformation
# ---------------------------------------------------------------------------

print("CRS ALIGNMENT")
print("-" * 80)
print(f"Storage source CRS: {storage_features.crs}")
print(f"Basemap target CRS: {target_crs}")


# ---------------------------------------------------------------------------
# Reproject storage features
# ---------------------------------------------------------------------------

storage_features_projected = storage_features.to_crs(target_crs)


# ---------------------------------------------------------------------------
# Post-transformation QA
# ---------------------------------------------------------------------------

print("\nPROJECTED STORAGE FEATURES")
print("-" * 80)

print(f"Rows:               {len(storage_features_projected):,}")
print(f"CRS:                {storage_features_projected.crs}")
print(f"Null geometries:     {storage_features_projected.geometry.isna().sum():,}")
print(f"Empty geometries:    {storage_features_projected.geometry.is_empty.sum():,}")
print(
    f"Invalid geometries:  "
    f"{(~storage_features_projected.geometry.is_valid).sum():,}"
)

print("\nGeometry types:")
print(
    storage_features_projected.geometry.geom_type.value_counts(
        dropna=False
    )
)


# ---------------------------------------------------------------------------
# Confirm CRS compatibility
# ---------------------------------------------------------------------------

assert storage_features_projected.crs == regions_25km.crs

print("\nStorage and basemap geometries now share the same CRS.")

CRS ALIGNMENT
--------------------------------------------------------------------------------
Storage source CRS: EPSG:3978
Basemap target CRS: EPSG:3347

PROJECTED STORAGE FEATURES
--------------------------------------------------------------------------------
Rows:               35,320
CRS:                EPSG:3347
Null geometries:     0
Empty geometries:    0
Invalid geometries:  0

Geometry types:
MultiPolygon          35295
Polygon                  24
GeometryCollection        1
Name: count, dtype: int64

Storage and basemap geometries now share the same CRS.


In [7]:
# ---------------------------------------------------------------------------
# Cell 8 — Build storage-feature → GeoCANOE-region intersection table
# ---------------------------------------------------------------------------

# Keep only the fields required for the exploratory spatial mapping.
region_fields = [
    "region",
    "site_id",
    "geometry",
]

storage_fields = [
    "storage_feature_id",
    "storage_unit_id",
    "source_dataset",
    "source_layer",
    "storage_type",
    "representation",
    "assessment_type",
    "data_class",
    "capacity_data",
    "injectivity_status",
    "geometry",
]


regions_overlay = regions_25km[region_fields].copy()
storage_overlay = storage_features_projected[storage_fields].copy()


# ---------------------------------------------------------------------------
# Preserve original geometry areas before intersection
# ---------------------------------------------------------------------------

regions_overlay["region_area_m2"] = regions_overlay.geometry.area
storage_overlay["storage_feature_area_m2"] = storage_overlay.geometry.area


# ---------------------------------------------------------------------------
# Spatial intersection
# ---------------------------------------------------------------------------

storage_region_intersections = gpd.overlay(
    regions_overlay,
    storage_overlay,
    how="intersection",
    keep_geom_type=False,
)


# ---------------------------------------------------------------------------
# Calculate intersection metrics
# ---------------------------------------------------------------------------

storage_region_intersections["intersection_area_m2"] = (
    storage_region_intersections.geometry.area
)

storage_region_intersections["region_overlap_fraction"] = (
    storage_region_intersections["intersection_area_m2"]
    / storage_region_intersections["region_area_m2"]
)

storage_region_intersections["feature_overlap_fraction"] = (
    storage_region_intersections["intersection_area_m2"]
    / storage_region_intersections["storage_feature_area_m2"]
)


# ---------------------------------------------------------------------------
# Initial QA
# ---------------------------------------------------------------------------

print("STORAGE → REGION INTERSECTIONS")
print("-" * 80)

print(
    f"Intersection records: "
    f"{len(storage_region_intersections):,}"
)

print(
    f"GeoCANOE regions intersecting storage: "
    f"{storage_region_intersections['region'].nunique():,} "
    f"of {len(regions_25km):,}"
)

print(
    f"Storage features intersecting GeoCANOE regions: "
    f"{storage_region_intersections['storage_feature_id'].nunique():,} "
    f"of {len(storage_features_projected):,}"
)

print("\nIntersections by source:")
print(
    storage_region_intersections["source_dataset"]
    .value_counts(dropna=False)
)

print("\nRegion overlap fraction:")
print(
    storage_region_intersections["region_overlap_fraction"]
    .describe()
)

print("\nFeature overlap fraction:")
print(
    storage_region_intersections["feature_overlap_fraction"]
    .describe()
)

STORAGE → REGION INTERSECTIONS
--------------------------------------------------------------------------------
Intersection records: 61,748
GeoCANOE regions intersecting storage: 2,735 of 9,269
Storage features intersecting GeoCANOE regions: 30,637 of 35,320

Intersections by source:
source_dataset
NATCARB             57875
BC_STORAGE_ATLAS     2332
ATLANTIC_COS         1541
Name: count, dtype: int64

Region overlap fraction:
count    6.174800e+04
mean     7.592483e-02
std      7.742266e-02
min      1.792617e-09
25%      1.736208e-02
50%      6.157542e-02
75%      1.291884e-01
max      1.000000e+00
Name: region_overlap_fraction, dtype: float64

Feature overlap fraction:
count    6.174800e+04
mean     4.829285e-01
std      3.652884e-01
min      1.156370e-08
25%      1.311246e-01
50%      4.299933e-01
75%      8.643069e-01
max      1.000000e+00
Name: feature_overlap_fraction, dtype: float64


In [8]:
# ---------------------------------------------------------------------------
# Cell 9 — Calculate unioned storage coverage by region and source
# ---------------------------------------------------------------------------

# Dissolve storage-region intersection geometries by GeoCANOE region and
# source dataset.
#
# This removes geometric double-counting where multiple spatial storage
# features from the same source overlap within a region and allows the
# fraction of each GeoCANOE region covered by that source to be calculated.
#
# IMPORTANT:
# This is a spatial coverage aggregation only. Storage assessments and
# capacities are not aggregated here. Assessments may be defined at either
# the individual storage-feature level or the logical storage-unit level,
# and repeated geometries may represent the same underlying storage unit.
# Capacity reconciliation is therefore handled separately.

source_region_coverage = (
    storage_region_intersections[
        [
            "region",
            "site_id",
            "source_dataset",
            "region_area_m2",
            "geometry",
        ]
    ]
    .dissolve(
        by=["region", "site_id", "source_dataset"],
        as_index=False,
    )
)


# ---------------------------------------------------------------------------
# Calculate unioned coverage
# ---------------------------------------------------------------------------

source_region_coverage["storage_union_area_m2"] = (
    source_region_coverage.geometry.area
)

source_region_coverage["storage_coverage_fraction"] = (
    source_region_coverage["storage_union_area_m2"]
    / source_region_coverage["region_area_m2"]
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print("UNIONED STORAGE COVERAGE BY REGION AND SOURCE")
print("-" * 80)

print(f"Records: {len(source_region_coverage):,}")
print(
    f"Unique GeoCANOE regions: "
    f"{source_region_coverage['region'].nunique():,}"
)

print("\nRegions represented by source:")
print(
    source_region_coverage.groupby("source_dataset")["region"]
    .nunique()
    .sort_values(ascending=False)
)


# ---------------------------------------------------------------------------
# Coverage distributions by source
# ---------------------------------------------------------------------------

coverage_summary = (
    source_region_coverage
    .groupby("source_dataset")["storage_coverage_fraction"]
    .describe()
)

print("\nStorage coverage fraction by source:")
display(coverage_summary)


# ---------------------------------------------------------------------------
# Inspect very small overlaps
# ---------------------------------------------------------------------------

thresholds = [0, 0.001, 0.01, 0.05, 0.10, 0.25, 0.50]

threshold_records = []

for source, group in source_region_coverage.groupby("source_dataset"):
    for threshold in thresholds:
        threshold_records.append(
            {
                "source_dataset": source,
                "minimum_coverage": threshold,
                "eligible_regions": (
                    group["storage_coverage_fraction"] > threshold
                ).sum(),
            }
        )

threshold_summary = pd.DataFrame(threshold_records)

print("\nSensitivity to minimum regional coverage:")
display(
    threshold_summary.pivot(
        index="minimum_coverage",
        columns="source_dataset",
        values="eligible_regions",
    )
)

UNIONED STORAGE COVERAGE BY REGION AND SOURCE
--------------------------------------------------------------------------------
Records: 2,892
Unique GeoCANOE regions: 2,735

Regions represented by source:
source_dataset
NATCARB             2579
BC_STORAGE_ATLAS     157
ATLANTIC_COS         156
Name: region, dtype: int64

Storage coverage fraction by source:


,count,mean,std,min,25%,50%,75%,max
source_dataset,,,,,,,,
ATLANTIC_COS,156.0,0.607979,0.388700,0.000010,0.209908,0.745833,0.997838,1.0
BC_STORAGE_ATLAS,157.0,0.698122,0.373307,0.000148,0.347551,0.932585,1.000000,1.0
NATCARB,2579.0,0.860521,0.289418,0.000013,0.965048,1.000000,1.000000,1.0



Sensitivity to minimum regional coverage:


source_dataset,ATLANTIC_COS,BC_STORAGE_ATLAS,NATCARB
minimum_coverage,,,
0.000,156,157,2579
0.001,155,154,2564
0.010,146,151,2535
0.050,134,145,2497
0.100,126,136,2440
0.250,114,125,2343
0.500,95,110,2219


In [9]:
# ---------------------------------------------------------------------------
# Cell 9.5 — Resolve storage assessments to spatial storage features
# ---------------------------------------------------------------------------

# Storage assessments may be defined at different semantic levels:
#
# - feature scope:
#     assessment → storage_feature_id
#
# - unit scope:
#     assessment → storage_unit_id → one or more spatial features
#
# Unit-scoped assessments are propagated to their associated spatial features
# for visualization and spatial analysis only. Their capacities must not be
# interpreted as additive where multiple geometries represent the same
# logical storage unit.


# ---------------------------------------------------------------------------
# Assessment fields required for spatial enrichment
# ---------------------------------------------------------------------------

assessment_fields = [
    "storage_assessment_id",
    "storage_feature_id",
    "storage_unit_id",
    "source_dataset",
    "assessment_scope",
    "storage_p10_tonnes",
    "storage_p50_tonnes",
    "storage_p90_tonnes",
    "theoretical_storage_tonnes",
    "effective_storage_tonnes",
]

assessments = storage_assessments[assessment_fields].copy()


# ---------------------------------------------------------------------------
# Resolve feature-scoped assessments
# ---------------------------------------------------------------------------

feature_assessments = assessments[
    assessments["assessment_scope"] == "feature"
].copy()

feature_resolved = (
    storage_features[
        [
            "storage_feature_id",
            "storage_unit_id",
            "source_dataset",
            "storage_type",
            "representation",
            "geometry",
        ]
    ]
    .merge(
        feature_assessments.drop(
            columns=[
                "storage_unit_id",
                "source_dataset",
            ]
        ),
        on="storage_feature_id",
        how="inner",
        validate="one_to_one",
    )
)


# ---------------------------------------------------------------------------
# Resolve unit-scoped assessments
# ---------------------------------------------------------------------------

unit_assessments = assessments[
    assessments["assessment_scope"] == "unit"
].copy()

unit_resolved = (
    storage_features[
        [
            "storage_feature_id",
            "storage_unit_id",
            "source_dataset",
            "storage_type",
            "representation",
            "geometry",
        ]
    ]
    .merge(
        unit_assessments.drop(
            columns=[
                "storage_feature_id",
                "source_dataset",
            ]
        ),
        on="storage_unit_id",
        how="inner",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------------------------
# Combine resolved feature representations
# ---------------------------------------------------------------------------

storage_features_assessed = gpd.GeoDataFrame(
    pd.concat(
        [
            feature_resolved,
            unit_resolved,
        ],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=storage_features.crs,
)


# ---------------------------------------------------------------------------
# Convert capacity fields to Mt CO2 for readable visualization
# ---------------------------------------------------------------------------

capacity_fields_tonnes = {
    "storage_p10_tonnes": "storage_p10_mt",
    "storage_p50_tonnes": "storage_p50_mt",
    "storage_p90_tonnes": "storage_p90_mt",
    "theoretical_storage_tonnes": "theoretical_storage_mt",
    "effective_storage_tonnes": "effective_storage_mt",
}

for tonnes_field, mt_field in capacity_fields_tonnes.items():
    storage_features_assessed[mt_field] = (
        storage_features_assessed[tonnes_field] / 1_000_000
    )


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print("RESOLVED STORAGE ASSESSMENTS")
print("-" * 80)

print(f"Resolved spatial features: {len(storage_features_assessed):,}")

print("\nResolved features by assessment scope:")
print(
    storage_features_assessed["assessment_scope"]
    .value_counts(dropna=False)
)

print("\nResolved features by source:")
print(
    storage_features_assessed["source_dataset"]
    .value_counts(dropna=False)
)

print("\nCapacity availability on resolved spatial features:")

display(
    storage_features_assessed
    .groupby("source_dataset")
    .agg(
        spatial_features=("storage_feature_id", "nunique"),
        logical_units=("storage_unit_id", "nunique"),
        p10_available=("storage_p10_mt", lambda x: x.notna().sum()),
        p50_available=("storage_p50_mt", lambda x: x.notna().sum()),
        p90_available=("storage_p90_mt", lambda x: x.notna().sum()),
        theoretical_available=(
            "theoretical_storage_mt",
            lambda x: x.notna().sum(),
        ),
        effective_available=(
            "effective_storage_mt",
            lambda x: x.notna().sum(),
        ),
    )
)

RESOLVED STORAGE ASSESSMENTS
--------------------------------------------------------------------------------
Resolved spatial features: 35,320

Resolved features by assessment scope:
assessment_scope
feature    33982
unit        1338
Name: count, dtype: int64

Resolved features by source:
source_dataset
NATCARB             29271
ATLANTIC_COS         4711
BC_STORAGE_ATLAS     1338
Name: count, dtype: int64

Capacity availability on resolved spatial features:


,spatial_features,logical_units,p10_available,p50_available,p90_available,theoretical_available,effective_available
source_dataset,,,,,,,
ATLANTIC_COS,4711,15,0,0,0,0,0
BC_STORAGE_ATLAS,1338,1263,29,29,29,1338,1309
NATCARB,29271,1565,23024,23024,23024,0,0


In [10]:
# ---------------------------------------------------------------------------
# Cell 10 — Interactive GeoCANOE × geological storage assessment map
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Prepare resolved assessment attributes for spatial crosswalk
# ---------------------------------------------------------------------------

# storage_features_assessed contains one resolved assessment representation
# per spatial storage feature:
#
#   feature-scoped assessment -> storage feature
#   unit-scoped assessment    -> propagated to associated storage features
#
# Keep assessment attributes only. Geometry comes from the GeoCANOE ×
# storage intersection table created in Cell 8.

assessment_map_fields = [
    "storage_feature_id",
    "storage_assessment_id",
    "assessment_scope",
    "storage_p10_mt",
    "storage_p50_mt",
    "storage_p90_mt",
    "theoretical_storage_mt",
    "effective_storage_mt",
]

assessment_map_lookup = (
    storage_features_assessed[assessment_map_fields]
    .copy()
)


# ---------------------------------------------------------------------------
# Join resolved assessments onto GeoCANOE × storage intersections
# ---------------------------------------------------------------------------

storage_region_map = (
    storage_region_intersections
    .merge(
        assessment_map_lookup,
        on="storage_feature_id",
        how="left",
        validate="many_to_one",
    )
)

storage_region_map = gpd.GeoDataFrame(
    storage_region_map,
    geometry="geometry",
    crs=storage_region_intersections.crs,
)


# ---------------------------------------------------------------------------
# Prepare human-readable visualization properties
# ---------------------------------------------------------------------------

source_display_names = {
    "NATCARB": "NATCARB",
    "BC_STORAGE_ATLAS": "BC Storage Atlas",
    "ATLANTIC_COS": "Atlantic COS",
}

storage_region_map["source_dataset"] = (
    storage_region_map["source_dataset"]
    .replace(source_display_names)
)


# Convert overlap fractions to percentages for display.

storage_region_map["region_overlap_percent"] = (
    storage_region_map["region_overlap_fraction"] * 100
).round(2)

storage_region_map["feature_overlap_percent"] = (
    storage_region_map["feature_overlap_fraction"] * 100
).round(2)


# Round capacity values for visualization only.

capacity_columns = [
    "storage_p10_mt",
    "storage_p50_mt",
    "storage_p90_mt",
    "theoretical_storage_mt",
    "effective_storage_mt",
]

storage_region_map[capacity_columns] = (
    storage_region_map[capacity_columns]
    .round(3)
)


# ---------------------------------------------------------------------------
# Keep only properties useful for interactive inspection
# ---------------------------------------------------------------------------

storage_region_map = storage_region_map[
    [
        "region",
        "site_id",
        "storage_feature_id",
        "storage_unit_id",
        "storage_assessment_id",
        "source_dataset",
        "storage_type",
        "representation",
        "assessment_scope",
        "region_overlap_percent",
        "feature_overlap_percent",
        "storage_p10_mt",
        "storage_p50_mt",
        "storage_p90_mt",
        "theoretical_storage_mt",
        "effective_storage_mt",
        "geometry",
    ]
].copy()


storage_region_map = storage_region_map.rename(
    columns={
        "region": "GeoCANOE Region",
        "site_id": "Site ID",
        "storage_feature_id": "Feature ID",
        "storage_unit_id": "Unit ID",
        "storage_assessment_id": "Assessment ID",
        "source_dataset": "Source",
        "storage_type": "Storage Type",
        "representation": "Representation",
        "assessment_scope": "Assessment Scope",
        "region_overlap_percent": "Region Overlap (%)",
        "feature_overlap_percent": "Feature Overlap (%)",
        "storage_p10_mt": "P10 Capacity (Mt)",
        "storage_p50_mt": "P50 Capacity (Mt)",
        "storage_p90_mt": "P90 Capacity (Mt)",
        "theoretical_storage_mt": "Theoretical (Mt)",
        "effective_storage_mt": "Effective (Mt)",
    }
)


# ---------------------------------------------------------------------------
# Normalize exceptional geometries for Lonboard
# ---------------------------------------------------------------------------

def extract_polygonal_geometry(geometry):
    """Return Polygon/MultiPolygon components from a geometry."""

    if geometry is None or geometry.is_empty:
        return None

    if geometry.geom_type in {"Polygon", "MultiPolygon"}:
        return geometry

    if geometry.geom_type == "GeometryCollection":
        polygon_parts = [
            part
            for part in geometry.geoms
            if part.geom_type in {"Polygon", "MultiPolygon"}
        ]

        if not polygon_parts:
            return None

        return unary_union(polygon_parts)

    return None


geometry_collections = (
    storage_region_map.geometry.geom_type == "GeometryCollection"
)

print(
    f"GeometryCollections requiring conversion: "
    f"{geometry_collections.sum():,}"
)

storage_region_map.loc[geometry_collections, "geometry"] = (
    storage_region_map.loc[geometry_collections, "geometry"]
    .apply(extract_polygonal_geometry)
)

storage_region_map = storage_region_map[
    storage_region_map.geometry.notna()
    & ~storage_region_map.geometry.is_empty
].copy()


# ---------------------------------------------------------------------------
# Split GeoCANOE × storage intersections by source
# ---------------------------------------------------------------------------

natcarb_map = storage_region_map[
    storage_region_map["Source"] == "NATCARB"
].copy()

bc_map = storage_region_map[
    storage_region_map["Source"] == "BC Storage Atlas"
].copy()

atlantic_map = storage_region_map[
    storage_region_map["Source"] == "Atlantic COS"
].copy()


# ---------------------------------------------------------------------------
# GeoCANOE grid reference layer
# ---------------------------------------------------------------------------

regions_map = regions_25km[
    [
        "region",
        "site_id",
        "geometry",
    ]
].copy()

regions_map = regions_map.rename(
    columns={
        "region": "GeoCANOE Region",
        "site_id": "Site ID",
    }
)

regions_layer = PolygonLayer.from_geopandas(
    regions_map,
    get_fill_color=[235, 235, 235, 15],
    get_line_color=[100, 100, 100, 100],
    line_width_min_pixels=0.5,
)


# ---------------------------------------------------------------------------
# GeoCANOE × geological storage layers
# ---------------------------------------------------------------------------

natcarb_layer = PolygonLayer.from_geopandas(
    natcarb_map,
    get_fill_color=[33, 102, 172, 100],
    get_line_color=[33, 102, 172, 180],
    line_width_min_pixels=0.5,
)

bc_layer = PolygonLayer.from_geopandas(
    bc_map,
    get_fill_color=[26, 152, 80, 120],
    get_line_color=[26, 152, 80, 200],
    line_width_min_pixels=0.75,
)

atlantic_layer = PolygonLayer.from_geopandas(
    atlantic_map,
    get_fill_color=[215, 48, 39, 100],
    get_line_color=[215, 48, 39, 180],
    line_width_min_pixels=0.5,
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print("\nGEOCANOE × STORAGE MAP")
print("-" * 80)

print(f"Intersection records: {len(storage_region_map):,}")
print(
    f"GeoCANOE regions represented: "
    f"{storage_region_map['GeoCANOE Region'].nunique():,}"
)
print(
    f"Storage features represented: "
    f"{storage_region_map['Feature ID'].nunique():,}"
)

print("\nRecords by source:")
print(
    storage_region_map["Source"]
    .value_counts()
)


# ---------------------------------------------------------------------------
# Build interactive map
# ---------------------------------------------------------------------------

storage_map_view = Map(
    layers=[
        regions_layer,
        natcarb_layer,
        bc_layer,
        atlantic_layer,
    ]
)

storage_map_view

GeometryCollections requiring conversion: 1


c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(
c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(



GEOCANOE × STORAGE MAP
--------------------------------------------------------------------------------
Intersection records: 61,748
GeoCANOE regions represented: 2,735
Storage features represented: 30,637

Records by source:
Source
NATCARB             57875
BC Storage Atlas     2332
Atlantic COS         1541
Name: count, dtype: int64


c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(
c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(


## Checkpoint — Geological Storage Mapped to GeoCANOE Regions

The unified Canadian geological CO₂ storage database has now been successfully
mapped onto the 25 km GeoCANOE `provinces_only` basemap.

### Spatial mapping

The workflow currently preserves three distinct representations:

1. **Storage features** — source-derived spatial representations of geological
   storage resources or prospectivity.
2. **GeoCANOE regions** — 25 km model regions identified by `region` and
   `site_id`.
3. **Storage-region intersections** — the many-to-many spatial relationship
   between geological storage features and GeoCANOE regions.

For each storage-region intersection, the workflow retains:

- GeoCANOE `region` and `site_id`
- storage feature and logical unit identifiers
- source dataset
- storage type and spatial representation
- assessment scope
- regional and feature overlap fractions
- available geological storage-capacity estimates

This provides an explicit crosswalk:

**storage source → storage feature/unit → GeoCANOE region → model-ready parameter**

### Assessment resolution

The unified database contains assessments defined at different semantic levels.

- **NATCARB** assessments are primarily feature-scoped.
- **BC Storage Atlas** assessments are unit-scoped and may therefore correspond
  to multiple spatial representations.
- **Atlantic COS** observations are feature-scoped qualitative prospectivity
  assessments and do not currently provide quantitative storage capacity.

Unit-scoped assessments are propagated to their associated spatial features for
mapping and inspection. Repeated spatial representations of the same logical
storage unit must **not** be interpreted as independent or additive storage
capacity.

### Interactive validation

The Lonboard visualization now displays the actual intersection between storage
features and GeoCANOE regions.

Selecting a mapped storage feature provides both:

- the associated GeoCANOE `Site ID`, and
- the underlying geological storage assessment and provenance.

Where available, P10, P50, P90, theoretical, and effective storage estimates are
displayed in Mt CO₂. These values remain properties of the underlying geological
assessment.

### Important modeling distinction

No geological capacity has yet been allocated to individual GeoCANOE regions.

If one geological storage feature intersects multiple GeoCANOE regions, its
capacity may appear in multiple intersection records for spatial inspection.
Therefore:

**displayed assessment capacity ≠ regional storage capacity**

and capacities must not yet be summed across storage-region intersection
records.

Similarly, overlapping NATCARB and BC Storage Atlas observations may describe
the same or overlapping geological resources. Source overlap must be reconciled
before quantitative regional capacities are constructed.

### Current model-ready interpretation

For the initial GeoCANOE implementation, the safest parameter that can already
be derived is binary geological storage accessibility:

\[
A_r =
\begin{cases}
1, & \text{if region } r \text{ contains eligible geological storage} \\
0, & \text{otherwise}
\end{cases}
\]

This supports an initial effectively unconstrained-storage formulation without
requiring premature assumptions about capacity allocation or source
reconciliation.

### Next steps

The next stage is to move from the spatial crosswalk toward a formal regional
storage representation by:

- defining storage eligibility criteria;
- examining NATCARB–BC source overlap;
- determining source precedence or reconciliation rules;
- distinguishing qualitative prospectivity from quantitative capacity;
- defining how storage capacity should be assigned to GeoCANOE regions; and
- eventually incorporating finite capacity, injectivity, and injection cost.

The detailed storage-region crosswalk should be retained even when the initial
optimization uses only binary storage accessibility, because it provides the
foundation for progressively richer geological storage constraints.

In [11]:
# ---------------------------------------------------------------------------
# Cell 11 — Classify GeoCANOE regions by geological storage source coverage
# ---------------------------------------------------------------------------

# Determine which storage datasets are represented in each GeoCANOE region
# using the unioned source-region coverage table from Cell 9.
#
# This is a source-presence diagnostic only. It does not reconcile overlapping
# geological resources or aggregate storage capacity.


# ---------------------------------------------------------------------------
# Create region × source presence table
# ---------------------------------------------------------------------------

source_presence = (
    source_region_coverage[
        [
            "region",
            "site_id",
            "source_dataset",
            "storage_coverage_fraction",
        ]
    ]
    .copy()
)

source_presence["source_present"] = True


# ---------------------------------------------------------------------------
# Pivot source presence to one row per GeoCANOE region
# ---------------------------------------------------------------------------

region_source_matrix = (
    source_presence
    .pivot_table(
        index=["region", "site_id"],
        columns="source_dataset",
        values="source_present",
        aggfunc="any",
        fill_value=False,
    )
    .reset_index()
)

region_source_matrix.columns.name = None


# Ensure expected source columns exist even if a future subset omits one.

expected_sources = [
    "NATCARB",
    "BC_STORAGE_ATLAS",
    "ATLANTIC_COS",
]

for source in expected_sources:
    if source not in region_source_matrix.columns:
        region_source_matrix[source] = False


# ---------------------------------------------------------------------------
# Create human-readable source combination
# ---------------------------------------------------------------------------

source_labels = {
    "NATCARB": "NATCARB",
    "BC_STORAGE_ATLAS": "BC Storage Atlas",
    "ATLANTIC_COS": "Atlantic COS",
}


def classify_source_combination(row):
    """Return the geological storage sources represented in a region."""

    present = [
        source_labels[source]
        for source in expected_sources
        if row[source]
    ]

    return " + ".join(present) if present else "None"


region_source_matrix["source_combination"] = (
    region_source_matrix.apply(
        classify_source_combination,
        axis=1,
    )
)

region_source_matrix["source_count"] = (
    region_source_matrix[expected_sources]
    .sum(axis=1)
)


# ---------------------------------------------------------------------------
# QA — source combinations
# ---------------------------------------------------------------------------

print("GEOCANOE STORAGE SOURCE COMBINATIONS")
print("-" * 80)

combination_summary = (
    region_source_matrix["source_combination"]
    .value_counts()
    .rename_axis("source_combination")
    .reset_index(name="regions")
)

display(combination_summary)


print("\nNumber of storage sources represented per region:")

display(
    region_source_matrix["source_count"]
    .value_counts()
    .sort_index()
    .rename_axis("source_count")
    .reset_index(name="regions")
)


# ---------------------------------------------------------------------------
# Focus on NATCARB × BC Storage Atlas overlap
# ---------------------------------------------------------------------------

natcarb_bc_overlap = region_source_matrix[
    region_source_matrix["NATCARB"]
    & region_source_matrix["BC_STORAGE_ATLAS"]
].copy()

print(
    "\nGeoCANOE regions containing both NATCARB "
    "and BC Storage Atlas:"
)
print(f"{len(natcarb_bc_overlap):,}")


# ---------------------------------------------------------------------------
# Attach source-specific coverage fractions for overlapping regions
# ---------------------------------------------------------------------------

coverage_wide = (
    source_presence
    .pivot_table(
        index=["region", "site_id"],
        columns="source_dataset",
        values="storage_coverage_fraction",
        aggfunc="max",
    )
    .reset_index()
)

coverage_wide.columns.name = None

natcarb_bc_overlap = (
    natcarb_bc_overlap
    .merge(
        coverage_wide,
        on=["region", "site_id"],
        how="left",
        suffixes=("", "_coverage"),
    )
)


# ---------------------------------------------------------------------------
# Inspect NATCARB × BC overlap coverage
# ---------------------------------------------------------------------------

if len(natcarb_bc_overlap) > 0:

    overlap_coverage_summary = (
        natcarb_bc_overlap[
            [
                "NATCARB_coverage",
                "BC_STORAGE_ATLAS_coverage",
            ]
        ]
        .describe()
    )

    print("\nCoverage fractions in NATCARB × BC overlap regions:")
    display(overlap_coverage_summary)

GEOCANOE STORAGE SOURCE COMBINATIONS
--------------------------------------------------------------------------------


,source_combination,regions
0,NATCARB,2422
1,NATCARB + BC Storage Atlas,157
2,Atlantic COS,156



Number of storage sources represented per region:


,source_count,regions
0,1,2578
1,2,157



GeoCANOE regions containing both NATCARB and BC Storage Atlas:
157

Coverage fractions in NATCARB × BC overlap regions:


,NATCARB_coverage,BC_STORAGE_ATLAS_coverage
count,157.000000,157.000000
mean,0.993757,0.698122
std,0.043096,0.373307
min,0.504151,0.000148
25%,1.000000,0.347551
50%,1.000000,0.932585
75%,1.000000,1.000000
max,1.000000,1.000000


In [12]:
# ---------------------------------------------------------------------------
# Cell 12 — Quantify NATCARB × BC spatial overlap within GeoCANOE regions
# ---------------------------------------------------------------------------

# Cell 11 identified 157 GeoCANOE regions containing both NATCARB and the
# BC Storage Atlas.
#
# Here we move from source co-occurrence within the same GeoCANOE region to
# actual geometric coincidence:
#
#     NATCARB footprint ∩ BC Storage Atlas footprint
#
# The source footprints were already dissolved within each region in Cell 9,
# so overlapping features from the same source are not double-counted.


# ---------------------------------------------------------------------------
# Extract NATCARB and BC source-region footprints
# ---------------------------------------------------------------------------

natcarb_coverage = (
    source_region_coverage[
        source_region_coverage["source_dataset"] == "NATCARB"
    ][
        [
            "region",
            "site_id",
            "storage_union_area_m2",
            "storage_coverage_fraction",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "storage_union_area_m2": "natcarb_area_m2",
            "storage_coverage_fraction": "natcarb_region_fraction",
        }
    )
    .copy()
)

bc_coverage = (
    source_region_coverage[
        source_region_coverage["source_dataset"] == "BC_STORAGE_ATLAS"
    ][
        [
            "region",
            "site_id",
            "storage_union_area_m2",
            "storage_coverage_fraction",
            "geometry",
        ]
    ]
    .rename(
        columns={
            "storage_union_area_m2": "bc_area_m2",
            "storage_coverage_fraction": "bc_region_fraction",
        }
    )
    .copy()
)


# ---------------------------------------------------------------------------
# Restrict analysis to the 157 known NATCARB × BC regions
# ---------------------------------------------------------------------------

overlap_region_ids = natcarb_bc_overlap[
    ["region", "site_id"]
].drop_duplicates()

natcarb_overlap_regions = natcarb_coverage.merge(
    overlap_region_ids,
    on=["region", "site_id"],
    how="inner",
)

bc_overlap_regions = bc_coverage.merge(
    overlap_region_ids,
    on=["region", "site_id"],
    how="inner",
)


# ---------------------------------------------------------------------------
# Pair the two dissolved source footprints within each GeoCANOE region
# ---------------------------------------------------------------------------

natcarb_bc_geometry = natcarb_overlap_regions.merge(
    bc_overlap_regions,
    on=["region", "site_id"],
    how="inner",
    suffixes=("_natcarb", "_bc"),
    validate="one_to_one",
)


# ---------------------------------------------------------------------------
# Calculate actual NATCARB × BC geometric intersection
# ---------------------------------------------------------------------------

natcarb_bc_geometry["overlap_geometry"] = [
    nat_geom.intersection(bc_geom)
    for nat_geom, bc_geom in zip(
        natcarb_bc_geometry["geometry_natcarb"],
        natcarb_bc_geometry["geometry_bc"],
    )
]

natcarb_bc_geometry["overlap_area_m2"] = (
    natcarb_bc_geometry["overlap_geometry"]
    .apply(lambda geom: geom.area if geom is not None else 0.0)
)


# ---------------------------------------------------------------------------
# Calculate overlap fractions
# ---------------------------------------------------------------------------

# Fraction of the BC footprint that is also covered by NATCARB.
natcarb_bc_geometry["bc_covered_by_natcarb_fraction"] = (
    natcarb_bc_geometry["overlap_area_m2"]
    / natcarb_bc_geometry["bc_area_m2"]
)

# Fraction of the NATCARB footprint that is also covered by BC.
natcarb_bc_geometry["natcarb_covered_by_bc_fraction"] = (
    natcarb_bc_geometry["overlap_area_m2"]
    / natcarb_bc_geometry["natcarb_area_m2"]
)

# Fraction of the complete GeoCANOE region occupied by both sources.
region_area_lookup = (
    regions_25km[
        ["region", "site_id", "geometry"]
    ]
    .copy()
)

region_area_lookup["region_area_m2"] = (
    region_area_lookup.geometry.area
)

natcarb_bc_geometry = natcarb_bc_geometry.merge(
    region_area_lookup[
        ["region", "site_id", "region_area_m2"]
    ],
    on=["region", "site_id"],
    how="left",
    validate="one_to_one",
)

natcarb_bc_geometry["region_overlap_fraction"] = (
    natcarb_bc_geometry["overlap_area_m2"]
    / natcarb_bc_geometry["region_area_m2"]
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print("NATCARB × BC STORAGE ATLAS SPATIAL OVERLAP")
print("-" * 80)

print(
    f"Co-occurring GeoCANOE regions: "
    f"{len(natcarb_bc_geometry):,}"
)

print(
    f"Regions with positive physical overlap: "
    f"{(natcarb_bc_geometry['overlap_area_m2'] > 0).sum():,}"
)


# ---------------------------------------------------------------------------
# Summarize physical overlap
# ---------------------------------------------------------------------------

overlap_summary = (
    natcarb_bc_geometry[
        [
            "bc_covered_by_natcarb_fraction",
            "natcarb_covered_by_bc_fraction",
            "region_overlap_fraction",
        ]
    ]
    .describe()
)

print("\nSpatial overlap fractions:")
display(overlap_summary)


# ---------------------------------------------------------------------------
# Classify degree of BC coincidence with NATCARB
# ---------------------------------------------------------------------------

def classify_bc_overlap(fraction):
    """Classify the fraction of the BC footprint coincident with NATCARB."""

    if fraction == 0:
        return "No physical overlap"
    if fraction < 0.25:
        return "<25% of BC footprint"
    if fraction < 0.50:
        return "25–50% of BC footprint"
    if fraction < 0.90:
        return "50–90% of BC footprint"
    if fraction < 0.99:
        return "90–99% of BC footprint"

    return ">=99% of BC footprint"


natcarb_bc_geometry["bc_overlap_class"] = (
    natcarb_bc_geometry["bc_covered_by_natcarb_fraction"]
    .apply(classify_bc_overlap)
)

overlap_class_order = [
    "No physical overlap",
    "<25% of BC footprint",
    "25–50% of BC footprint",
    "50–90% of BC footprint",
    "90–99% of BC footprint",
    ">=99% of BC footprint",
]

overlap_class_summary = (
    natcarb_bc_geometry["bc_overlap_class"]
    .value_counts()
    .reindex(overlap_class_order, fill_value=0)
    .rename_axis("BC footprint overlap with NATCARB")
    .reset_index(name="regions")
)

print("\nBC footprint coincidence with NATCARB:")
display(overlap_class_summary)

NATCARB × BC STORAGE ATLAS SPATIAL OVERLAP
--------------------------------------------------------------------------------
Co-occurring GeoCANOE regions: 157
Regions with positive physical overlap: 157

Spatial overlap fractions:


,bc_covered_by_natcarb_fraction,natcarb_covered_by_bc_fraction,region_overlap_fraction
count,1.570000e+02,157.000000,157.000000
mean,1.000000e+00,0.699017,0.698122
std,2.473159e-15,0.372935,0.373307
min,1.000000e+00,0.000148,0.000148
25%,1.000000e+00,0.347551,0.347551
50%,1.000000e+00,0.932585,0.932585
75%,1.000000e+00,1.000000,1.000000
max,1.000000e+00,1.000000,1.000000



BC footprint coincidence with NATCARB:


,BC footprint overlap with NATCARB,regions
0,No physical overlap,0
1,<25% of BC footprint,0
2,25–50% of BC footprint,0
3,50–90% of BC footprint,0
4,90–99% of BC footprint,0
5,>=99% of BC footprint,157


In [13]:
# ---------------------------------------------------------------------------
# Cell 13 — Characterize NATCARB × BC geological storage-type overlap
# ---------------------------------------------------------------------------

# Cell 12 established that all mapped BC Storage Atlas footprints overlap
# NATCARB spatial coverage.
#
# This cell moves from source-level spatial coincidence to feature-level
# geological comparison by determining which BC storage types intersect
# which NATCARB storage types.
#
# Spatial overlap does not by itself imply that two features represent the
# same geological storage resource.


# ---------------------------------------------------------------------------
# Extract BC and NATCARB storage features
# ---------------------------------------------------------------------------

natcarb_features = storage_features_projected[
    storage_features_projected["source_dataset"] == "NATCARB"
][
    [
        "storage_feature_id",
        "storage_unit_id",
        "storage_type",
        "storage_subtype",
        "representation",
        "geometry",
    ]
].copy()

bc_features = storage_features_projected[
    storage_features_projected["source_dataset"] == "BC_STORAGE_ATLAS"
][
    [
        "storage_feature_id",
        "storage_unit_id",
        "storage_type",
        "storage_subtype",
        "representation",
        "geometry",
    ]
].copy()


# ---------------------------------------------------------------------------
# Rename fields before spatial join
# ---------------------------------------------------------------------------

natcarb_features = natcarb_features.rename(
    columns={
        "storage_feature_id": "natcarb_feature_id",
        "storage_unit_id": "natcarb_unit_id",
        "storage_type": "natcarb_storage_type",
        "storage_subtype": "natcarb_storage_subtype",
        "representation": "natcarb_representation",
    }
)

bc_features = bc_features.rename(
    columns={
        "storage_feature_id": "bc_feature_id",
        "storage_unit_id": "bc_unit_id",
        "storage_type": "bc_storage_type",
        "storage_subtype": "bc_storage_subtype",
        "representation": "bc_representation",
    }
)


# ---------------------------------------------------------------------------
# Identify candidate feature pairs using spatial indexing
# ---------------------------------------------------------------------------

# Use a spatial join first rather than overlaying every possible feature pair.
# This returns BC × NATCARB feature pairs whose geometries intersect.

feature_pairs = gpd.sjoin(
    bc_features,
    natcarb_features,
    how="inner",
    predicate="intersects",
    lsuffix="bc",
    rsuffix="natcarb",
)

feature_pairs = feature_pairs.reset_index(drop=True)


# ---------------------------------------------------------------------------
# QA — number of intersecting feature pairs
# ---------------------------------------------------------------------------

print("NATCARB × BC FEATURE-LEVEL OVERLAP")
print("-" * 80)

print(f"Intersecting feature pairs: {len(feature_pairs):,}")

print(
    f"BC features intersecting NATCARB: "
    f"{feature_pairs['bc_feature_id'].nunique():,} "
    f"of {len(bc_features):,}"
)

print(
    f"NATCARB features intersecting BC: "
    f"{feature_pairs['natcarb_feature_id'].nunique():,} "
    f"of {len(natcarb_features):,}"
)


# ---------------------------------------------------------------------------
# Summarize storage-type combinations
# ---------------------------------------------------------------------------

type_overlap_summary = (
    feature_pairs
    .groupby(
        [
            "bc_storage_type",
            "natcarb_storage_type",
        ],
        dropna=False,
    )
    .agg(
        feature_pairs=("bc_feature_id", "size"),
        bc_features=("bc_feature_id", "nunique"),
        bc_units=("bc_unit_id", "nunique"),
        natcarb_features=("natcarb_feature_id", "nunique"),
        natcarb_units=("natcarb_unit_id", "nunique"),
    )
    .reset_index()
    .sort_values(
        "feature_pairs",
        ascending=False,
    )
)

print("\nBC × NATCARB storage-type combinations:")
display(type_overlap_summary)


# ---------------------------------------------------------------------------
# Summarize representation combinations
# ---------------------------------------------------------------------------

representation_overlap_summary = (
    feature_pairs
    .groupby(
        [
            "bc_representation",
            "natcarb_representation",
        ],
        dropna=False,
    )
    .agg(
        feature_pairs=("bc_feature_id", "size"),
        bc_features=("bc_feature_id", "nunique"),
        natcarb_features=("natcarb_feature_id", "nunique"),
    )
    .reset_index()
    .sort_values(
        "feature_pairs",
        ascending=False,
    )
)

print("\nBC × NATCARB representation combinations:")
display(representation_overlap_summary)


# ---------------------------------------------------------------------------
# Number of NATCARB storage types encountered by each BC feature
# ---------------------------------------------------------------------------

bc_type_diversity = (
    feature_pairs
    .groupby("bc_feature_id")["natcarb_storage_type"]
    .nunique()
)

print("\nNumber of NATCARB storage types intersecting each BC feature:")
display(
    bc_type_diversity
    .value_counts()
    .sort_index()
    .rename_axis("NATCARB storage types")
    .reset_index(name="BC features")
)

NATCARB × BC FEATURE-LEVEL OVERLAP
--------------------------------------------------------------------------------
Intersecting feature pairs: 13,073
BC features intersecting NATCARB: 1,338 of 1,338
NATCARB features intersecting BC: 1,834 of 29,271

BC × NATCARB storage-type combinations:


,bc_storage_type,natcarb_storage_type,feature_pairs,bc_features,bc_units,natcarb_features,natcarb_units
2,depleted_hydrocarbon_reservoir,saline_aquifer,4754,1309,1238,981,8
5,saline_aquifer,saline_aquifer,3268,29,25,1223,7
1,depleted_hydrocarbon_reservoir,depleted_hydrocarbon_reservoir,2786,1267,1196,285,285
4,saline_aquifer,depleted_hydrocarbon_reservoir,1074,29,25,275,275
3,saline_aquifer,coal,641,16,14,138,1
0,depleted_hydrocarbon_reservoir,coal,550,262,252,136,1



BC × NATCARB representation combinations:


,bc_representation,natcarb_representation,feature_pairs,bc_features,natcarb_features
2,pool_extent,resource_grid_cell,5304,1309,1117
0,aquifer_extent,resource_grid_cell,3909,29,1361
3,pool_extent,storage_resource,2786,1267,285
1,aquifer_extent,storage_resource,1074,29,275



Number of NATCARB storage types intersecting each BC feature:


,NATCARB storage types,BC features
0,1,33
1,2,1036
2,3,269


In [14]:
# ---------------------------------------------------------------------------
# Cell 14 — Inspect candidate BC × NATCARB resource identity relationships
# ---------------------------------------------------------------------------

# Cell 13 showed that spatial coincidence is strongly many-to-many and that
# broad NATCARB resource-grid coverage intersects several BC storage classes.
#
# This cell focuses on the most plausible reconciliation case:
#
#     BC depleted hydrocarbon reservoir
#         ×
#     NATCARB depleted hydrocarbon reservoir
#
# These same-type intersections are candidates for further identity analysis,
# but are NOT assumed to represent duplicate geological resources.
#
# We also inspect the BC features for which the Cell 13 storage-type diversity
# diagnostic returned zero because pandas nunique() excludes missing values.


# ---------------------------------------------------------------------------
# 1. Resolve the apparent zero-type diagnostic from Cell 13
# ---------------------------------------------------------------------------

bc_zero_type_ids = bc_type_diversity[
    bc_type_diversity == 0
].index

zero_type_pairs = feature_pairs[
    feature_pairs["bc_feature_id"].isin(bc_zero_type_ids)
].copy()

print("NULL STORAGE-TYPE DIAGNOSTIC")
print("-" * 80)

print(f"BC features flagged with zero NATCARB storage types: {len(bc_zero_type_ids):,}")
print(f"Associated BC × NATCARB feature pairs: {len(zero_type_pairs):,}")

print("\nNATCARB storage type values in these pairs:")
display(
    zero_type_pairs["natcarb_storage_type"]
    .value_counts(dropna=False)
    .rename_axis("natcarb_storage_type")
    .reset_index(name="feature_pairs")
)

print("\nNATCARB representations in these pairs:")
display(
    zero_type_pairs["natcarb_representation"]
    .value_counts(dropna=False)
    .rename_axis("natcarb_representation")
    .reset_index(name="feature_pairs")
)


# ---------------------------------------------------------------------------
# 2. Isolate same-type depleted-hydrocarbon intersections
# ---------------------------------------------------------------------------

depleted_pairs = feature_pairs[
    (
        feature_pairs["bc_storage_type"]
        == "depleted_hydrocarbon_reservoir"
    )
    &
    (
        feature_pairs["natcarb_storage_type"]
        == "depleted_hydrocarbon_reservoir"
    )
].copy()

print("\nSAME-TYPE DEPLETED HYDROCARBON INTERSECTIONS")
print("-" * 80)

print(f"Feature pairs: {len(depleted_pairs):,}")
print(f"Unique BC features: {depleted_pairs['bc_feature_id'].nunique():,}")
print(f"Unique BC units: {depleted_pairs['bc_unit_id'].nunique():,}")
print(
    f"Unique NATCARB features: "
    f"{depleted_pairs['natcarb_feature_id'].nunique():,}"
)
print(
    f"Unique NATCARB units: "
    f"{depleted_pairs['natcarb_unit_id'].nunique():,}"
)


# ---------------------------------------------------------------------------
# 3. Inspect canonical storage-unit identity fields
# ---------------------------------------------------------------------------

identity_fields = [
    "storage_unit_id",
    "source_dataset",
    "source_layer",
    "source_unit_id",
    "storage_type",
    "storage_subtype",
    "storage_name",
    "formation_name",
    "group_name",
    "basin_name",
    "province_territory",
]

available_identity_fields = [
    column
    for column in identity_fields
    if column in storage_units.columns
]

print("\nAvailable canonical identity fields:")
print(available_identity_fields)


# ---------------------------------------------------------------------------
# 4. Build BC and NATCARB unit lookup tables
# ---------------------------------------------------------------------------

bc_unit_lookup = (
    storage_units[
        storage_units["source_dataset"] == "BC_STORAGE_ATLAS"
    ][available_identity_fields]
    .drop_duplicates(subset=["storage_unit_id"])
    .copy()
)

natcarb_unit_lookup = (
    storage_units[
        storage_units["source_dataset"] == "NATCARB"
    ][available_identity_fields]
    .drop_duplicates(subset=["storage_unit_id"])
    .copy()
)


# Prefix source-specific attributes before joining.

bc_unit_lookup = bc_unit_lookup.rename(
    columns={
        column: f"bc_{column}"
        for column in bc_unit_lookup.columns
        if column != "storage_unit_id"
    }
).rename(
    columns={"storage_unit_id": "bc_unit_id"}
)

natcarb_unit_lookup = natcarb_unit_lookup.rename(
    columns={
        column: f"natcarb_{column}"
        for column in natcarb_unit_lookup.columns
        if column != "storage_unit_id"
    }
).rename(
    columns={"storage_unit_id": "natcarb_unit_id"}
)


# ---------------------------------------------------------------------------
# 5. Attach logical-unit identity attributes to candidate pairs
# ---------------------------------------------------------------------------

depleted_pair_identity = (
    depleted_pairs
    .merge(
        bc_unit_lookup,
        on="bc_unit_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        natcarb_unit_lookup,
        on="natcarb_unit_id",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------------------------
# 6. Collapse feature-level intersections to unique logical-unit pairs
# ---------------------------------------------------------------------------

unit_pair_columns = [
    "bc_unit_id",
    "natcarb_unit_id",
]

descriptive_columns = [
    column
    for column in [
        "bc_source_unit_id",
        "bc_storage_name",
        "bc_formation_name",
        "bc_group_name",
        "bc_basin_name",
        "bc_province_territory",
        "natcarb_source_unit_id",
        "natcarb_storage_name",
        "natcarb_formation_name",
        "natcarb_group_name",
        "natcarb_basin_name",
        "natcarb_province_territory",
    ]
    if column in depleted_pair_identity.columns
]

depleted_unit_pairs = (
    depleted_pair_identity[
        unit_pair_columns + descriptive_columns
    ]
    .drop_duplicates(subset=unit_pair_columns)
    .reset_index(drop=True)
)

print("\nUnique candidate logical-unit pairs:")
print(f"{len(depleted_unit_pairs):,}")


# ---------------------------------------------------------------------------
# 7. Inspect availability of identity attributes
# ---------------------------------------------------------------------------

if descriptive_columns:

    identity_completeness = (
        depleted_unit_pairs[descriptive_columns]
        .notna()
        .sum()
        .rename("non_null_records")
        .to_frame()
    )

    identity_completeness["total_records"] = len(depleted_unit_pairs)

    identity_completeness["completeness_fraction"] = (
        identity_completeness["non_null_records"]
        / identity_completeness["total_records"]
    )

    print("\nIdentity-field completeness:")
    display(identity_completeness)


# ---------------------------------------------------------------------------
# 8. Preview candidate logical-unit relationships
# ---------------------------------------------------------------------------

print("\nCandidate BC × NATCARB depleted-reservoir unit pairs:")

display(
    depleted_unit_pairs.head(25)
)

NULL STORAGE-TYPE DIAGNOSTIC
--------------------------------------------------------------------------------
BC features flagged with zero NATCARB storage types: 0
Associated BC × NATCARB feature pairs: 0

NATCARB storage type values in these pairs:


,natcarb_storage_type,feature_pairs



NATCARB representations in these pairs:


,natcarb_representation,feature_pairs



SAME-TYPE DEPLETED HYDROCARBON INTERSECTIONS
--------------------------------------------------------------------------------
Feature pairs: 2,786
Unique BC features: 1,267
Unique BC units: 1,196
Unique NATCARB features: 285
Unique NATCARB units: 285

Available canonical identity fields:
['storage_unit_id', 'source_dataset', 'source_unit_id', 'storage_type', 'storage_subtype', 'storage_name', 'basin_name', 'province_territory']

Unique candidate logical-unit pairs:
2,558

Identity-field completeness:


,non_null_records,total_records,completeness_fraction
bc_source_unit_id,2558,2558,1.0
bc_storage_name,2558,2558,1.0
bc_basin_name,0,2558,0.0
bc_province_territory,2558,2558,1.0
natcarb_source_unit_id,2558,2558,1.0
natcarb_storage_name,2558,2558,1.0
natcarb_basin_name,0,2558,0.0
natcarb_province_territory,2558,2558,1.0



Candidate BC × NATCARB depleted-reservoir unit pairs:


,bc_unit_id,natcarb_unit_id,bc_source_unit_id,bc_storage_name,bc_basin_name,bc_province_territory,natcarb_source_unit_id,natcarb_storage_name,natcarb_basin_name,natcarb_province_territory
0,BC_POOL_gbc_pool_5000|4800B,NAT_OG_d1c4cc287176,gbc_pool_5000|4800B,Jedney Halfway B,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|JEDNEY,JEDNEY,None,BC
1,BC_POOL_gbc_pool_3540|4100E,NAT_OG_58b7e1332666,gbc_pool_3540|4100E,Fireweed Baldonnel E,None,British Columbia,NATCARB|PCOR|OIL_GAS|INGA|0.0,INGA,None,BC
2,BC_POOL_gbc_pool_3540|4100E,NAT_OG_229305a1c61f,gbc_pool_3540|4100E,Fireweed Baldonnel E,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|INGA,INGA,None,BC
3,BC_POOL_gbc_pool_3540|4100E,NAT_OG_6076cdb9d4c5,gbc_pool_3540|4100E,Fireweed Baldonnel E,None,British Columbia,NATCARB|PCOR|OIL_GAS|FIREWEED|0.0,FIREWEED,None,BC
4,BC_POOL_gbc_pool_3540|4100E,NAT_OG_29b7a9b47a50,gbc_pool_3540|4100E,Fireweed Baldonnel E,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|FIREWEED,FIREWEED,None,BC
5,BC_POOL_gbc_pool_7250|2600B,NAT_OG_71b32327e0e8,gbc_pool_7250|2600B,Pickell Bluesky B,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|PICKELL,PICKELL,None,BC
6,BC_POOL_gbc_pool_3320|2700C,NAT_OG_07a274996e1f,gbc_pool_3320|2700C,Currant West Gething C,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|CURRANT WEST,CURRANT WEST,None,BC
7,BC_POOL_gbc_pool_3320|2700C,NAT_OG_2b965683dd6d,gbc_pool_3320|2700C,Currant West Gething C,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|BUICK CREEK,BUICK CREEK,None,BC
8,BC_POOL_gbc_pool_6430|2505D,NAT_OG_17e658ef7cf7,gbc_pool_6430|2505D,Noel Falher A D,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|NOEL,NOEL,None,BC
9,BC_POOL_gbc_pool_6430|2510F,NAT_OG_3a7742391d11,gbc_pool_6430|2510F,Noel Falher B F,None,British Columbia,NATCARB|WESTCARB|OIL_GAS|KELLY,KELLY,None,BC


In [15]:
# ---------------------------------------------------------------------------
# Cell 15 — Test cross-source storage-name agreement
# ---------------------------------------------------------------------------

# Cell 14 showed that spatially intersecting same-type depleted-reservoir
# features often contain recognizable field-name relationships:
#
#     BC:      Jedney Halfway B
#     NATCARB: JEDNEY
#
# The BC storage name commonly contains:
#
#     field name + formation/pool information
#
# whereas the NATCARB name may represent the broader field.
#
# This cell therefore tests conservative lexical agreement between the
# candidate BC × NATCARB logical-unit pairs.
#
# IMPORTANT:
# Name agreement is evidence of possible cross-source identity, not proof
# that the two records represent the same geological storage resource.


# ---------------------------------------------------------------------------
# Normalize storage names
# ---------------------------------------------------------------------------

def normalize_storage_name(value):
    """Normalize a storage name for conservative lexical comparison."""

    if pd.isna(value):
        return None

    value = str(value).upper().strip()

    # Replace punctuation with spaces.
    value = re.sub(r"[^A-Z0-9]+", " ", value)

    # Collapse repeated whitespace.
    value = re.sub(r"\s+", " ", value).strip()

    return value


name_matches = depleted_unit_pairs.copy()

name_matches["bc_name_normalized"] = (
    name_matches["bc_storage_name"]
    .apply(normalize_storage_name)
)

name_matches["natcarb_name_normalized"] = (
    name_matches["natcarb_storage_name"]
    .apply(normalize_storage_name)
)


# ---------------------------------------------------------------------------
# Exact and token-level agreement
# ---------------------------------------------------------------------------

name_matches["exact_name_match"] = (
    name_matches["bc_name_normalized"]
    == name_matches["natcarb_name_normalized"]
)


def natcarb_name_in_bc_name(row):
    """Test whether the NATCARB field name occurs in the BC name."""

    bc_name = row["bc_name_normalized"]
    natcarb_name = row["natcarb_name_normalized"]

    if not bc_name or not natcarb_name:
        return False

    # Compare complete whitespace-delimited tokens rather than arbitrary
    # substrings so that short field names do not match inside other words.
    bc_tokens = bc_name.split()
    natcarb_tokens = natcarb_name.split()

    if len(natcarb_tokens) > len(bc_tokens):
        return False

    n = len(natcarb_tokens)

    return any(
        bc_tokens[i:i + n] == natcarb_tokens
        for i in range(len(bc_tokens) - n + 1)
    )


name_matches["natcarb_name_in_bc_name"] = (
    name_matches.apply(
        natcarb_name_in_bc_name,
        axis=1,
    )
)


# ---------------------------------------------------------------------------
# Conservative candidate identity flag
# ---------------------------------------------------------------------------

name_matches["candidate_name_match"] = (
    name_matches["exact_name_match"]
    | name_matches["natcarb_name_in_bc_name"]
)


# ---------------------------------------------------------------------------
# QA summary
# ---------------------------------------------------------------------------

print("BC × NATCARB STORAGE-NAME AGREEMENT")
print("-" * 80)

print(f"Candidate logical-unit pairs: {len(name_matches):,}")

print(
    f"Exact normalized-name matches: "
    f"{name_matches['exact_name_match'].sum():,}"
)

print(
    f"NATCARB name contained in BC name: "
    f"{name_matches['natcarb_name_in_bc_name'].sum():,}"
)

print(
    f"Conservative candidate name matches: "
    f"{name_matches['candidate_name_match'].sum():,}"
)


# ---------------------------------------------------------------------------
# How many BC units receive candidate NATCARB matches?
# ---------------------------------------------------------------------------

matched_pairs = name_matches[
    name_matches["candidate_name_match"]
].copy()

matched_bc_units = matched_pairs["bc_unit_id"].nunique()

print(
    f"\nBC depleted-reservoir units with >=1 candidate name match: "
    f"{matched_bc_units:,} "
    f"of {depleted_unit_pairs['bc_unit_id'].nunique():,}"
)


# ---------------------------------------------------------------------------
# Match multiplicity
# ---------------------------------------------------------------------------

match_multiplicity = (
    matched_pairs
    .groupby("bc_unit_id")
    ["natcarb_unit_id"]
    .nunique()
)

print("\nCandidate NATCARB matches per matched BC unit:")

display(
    match_multiplicity
    .value_counts()
    .sort_index()
    .rename_axis("candidate NATCARB units")
    .reset_index(name="BC units")
)


# ---------------------------------------------------------------------------
# Preview likely matches
# ---------------------------------------------------------------------------

preview_columns = [
    "bc_unit_id",
    "bc_storage_name",
    "natcarb_unit_id",
    "natcarb_storage_name",
    "exact_name_match",
    "natcarb_name_in_bc_name",
]

print("\nExample candidate name matches:")

display(
    matched_pairs[
        preview_columns
    ]
    .sort_values(
        [
            "bc_storage_name",
            "natcarb_storage_name",
        ]
    )
    .head(40)
)


# ---------------------------------------------------------------------------
# Preview spatial candidates rejected by conservative name matching
# ---------------------------------------------------------------------------

unmatched_pairs = name_matches[
    ~name_matches["candidate_name_match"]
].copy()

print("\nExample spatially intersecting pairs without name agreement:")

display(
    unmatched_pairs[
        preview_columns
    ]
    .sort_values(
        [
            "bc_storage_name",
            "natcarb_storage_name",
        ]
    )
    .head(40)
)

BC × NATCARB STORAGE-NAME AGREEMENT
--------------------------------------------------------------------------------
Candidate logical-unit pairs: 2,558
Exact normalized-name matches: 0
NATCARB name contained in BC name: 1,718
Conservative candidate name matches: 1,718

BC depleted-reservoir units with >=1 candidate name match: 1,167 of 1,196

Candidate NATCARB matches per matched BC unit:


,candidate NATCARB units,BC units
0,1,659
1,2,480
2,3,13
3,4,15



Example candidate name matches:


,bc_unit_id,bc_storage_name,natcarb_unit_id,natcarb_storage_name,exact_name_match,natcarb_name_in_bc_name
1152,BC_POOL_gbc_pool_0050|8400A,Adsett Slave Point A,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
1615,BC_POOL_gbc_pool_0050|8400B,Adsett Slave Point B,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
1614,BC_POOL_gbc_pool_0050|8400C,Adsett Slave Point C,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
1216,BC_POOL_gbc_pool_0050|8400H,Adsett Slave Point H,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
1215,BC_POOL_gbc_pool_0050|8400I,Adsett Slave Point I,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
745,BC_POOL_gbc_pool_0050|8400J,Adsett Slave Point J,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
2019,BC_POOL_gbc_pool_0050|8400L,Adsett Slave Point L,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
2020,BC_POOL_gbc_pool_0050|8400M,Adsett Slave Point M,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
1136,BC_POOL_gbc_pool_0050|8400N,Adsett Slave Point N,NAT_OG_b5d7a53c0fc9,ADSETT,False,True
634,BC_POOL_gbc_pool_0100|4100A,Airport Baldonnel A,NAT_OG_8528ef54c853,AIRPORT,False,True



Example spatially intersecting pairs without name agreement:


,bc_unit_id,bc_storage_name,natcarb_unit_id,natcarb_storage_name,exact_name_match,natcarb_name_in_bc_name
636,BC_POOL_gbc_pool_0100|4100A,Airport Baldonnel A,NAT_OG_36afd200150a,FORT ST JOHN,False,False
637,BC_POOL_gbc_pool_0100|4100A,Airport Baldonnel A,NAT_OG_e91f3016a6e6,FORT ST JOHN,False,False
2041,BC_POOL_gbc_pool_0100|2600A,Airport Bluesky A,NAT_OG_36afd200150a,FORT ST JOHN,False,False
2042,BC_POOL_gbc_pool_0100|2600A,Airport Bluesky A,NAT_OG_e91f3016a6e6,FORT ST JOHN,False,False
2039,BC_POOL_gbc_pool_0100|2600A,Airport Bluesky A,NAT_OG_787561b8c14b,FORT ST JOHN SOUTHEAST,False,False
632,BC_POOL_gbc_pool_0100|2900A,Airport Dunlevy A,NAT_OG_36afd200150a,FORT ST JOHN,False,False
633,BC_POOL_gbc_pool_0100|2900A,Airport Dunlevy A,NAT_OG_e91f3016a6e6,FORT ST JOHN,False,False
674,BC_POOL_gbc_pool_0400|4610A,Beatton River A Marker/Base Of Lime A,NAT_OG_ae2ccd2adacd,MILLIGAN CREEK,False,False
675,BC_POOL_gbc_pool_0400|4610A,Beatton River A Marker/Base Of Lime A,NAT_OG_64d1d15c075b,MILLIGAN CREEK,False,False
670,BC_POOL_gbc_pool_0400|4540A,Beatton River Coplin A,NAT_OG_ae2ccd2adacd,MILLIGAN CREEK,False,False


In [16]:
# ---------------------------------------------------------------------------
# Cell 16 — Build preliminary GeoCANOE geological storage eligibility
# ---------------------------------------------------------------------------

# The initial GeoCANOE storage formulation requires regional accessibility,
# not finite geological capacity.
#
# We therefore construct source/evidence flags first and then derive a
# preliminary binary storage-accessibility parameter:
#
#     A_r = 1  if region r contains eligible geological storage evidence
#           0  otherwise
#
# This deliberately avoids aggregating or reconciling storage capacities.
#
# IMPORTANT:
# Eligibility here means that geological storage evidence is spatially
# available within a GeoCANOE region. It does not imply project feasibility,
# injectivity, permitting, economic viability, or independently additive
# capacity across datasets.


# ---------------------------------------------------------------------------
# Start from the complete GeoCANOE basemap
# ---------------------------------------------------------------------------

regional_storage = regions_25km[
    [
        "region",
        "site_id",
        "province_codes",
        "geometry",
    ]
].copy()


# ---------------------------------------------------------------------------
# Build source-specific coverage fractions
# ---------------------------------------------------------------------------

coverage_by_source = (
    source_region_coverage[
        [
            "region",
            "site_id",
            "source_dataset",
            "storage_coverage_fraction",
        ]
    ]
    .pivot_table(
        index=["region", "site_id"],
        columns="source_dataset",
        values="storage_coverage_fraction",
        aggfunc="max",
        fill_value=0.0,
    )
    .reset_index()
)

coverage_by_source.columns.name = None


# Ensure expected source columns always exist.

expected_sources = [
    "NATCARB",
    "BC_STORAGE_ATLAS",
    "ATLANTIC_COS",
]

for source in expected_sources:
    if source not in coverage_by_source.columns:
        coverage_by_source[source] = 0.0


# ---------------------------------------------------------------------------
# Rename source-specific coverage fields
# ---------------------------------------------------------------------------

coverage_by_source = coverage_by_source.rename(
    columns={
        "NATCARB": "natcarb_coverage_fraction",
        "BC_STORAGE_ATLAS": "bc_coverage_fraction",
        "ATLANTIC_COS": "atlantic_coverage_fraction",
    }
)


# ---------------------------------------------------------------------------
# Attach coverage to every GeoCANOE region
# ---------------------------------------------------------------------------

regional_storage = regional_storage.merge(
    coverage_by_source,
    on=["region", "site_id"],
    how="left",
    validate="one_to_one",
)

coverage_columns = [
    "natcarb_coverage_fraction",
    "bc_coverage_fraction",
    "atlantic_coverage_fraction",
]

regional_storage[coverage_columns] = (
    regional_storage[coverage_columns]
    .fillna(0.0)
)


# ---------------------------------------------------------------------------
# Preserve evidence classes explicitly
# ---------------------------------------------------------------------------

regional_storage["has_natcarb"] = (
    regional_storage["natcarb_coverage_fraction"] > 0
)

regional_storage["has_bc_storage_atlas"] = (
    regional_storage["bc_coverage_fraction"] > 0
)

regional_storage["has_atlantic_cos"] = (
    regional_storage["atlantic_coverage_fraction"] > 0
)


# Quantitative capacity evidence currently comes from NATCARB and BC.
regional_storage["has_quantitative_storage_evidence"] = (
    regional_storage["has_natcarb"]
    | regional_storage["has_bc_storage_atlas"]
)


# Atlantic COS is retained separately because it represents qualitative
# geological prospectivity rather than quantified storage capacity.
regional_storage["has_qualitative_storage_evidence"] = (
    regional_storage["has_atlantic_cos"]
)


# ---------------------------------------------------------------------------
# Preliminary binary accessibility parameter
# ---------------------------------------------------------------------------

# For the initial unconstrained-storage formulation, accept either
# quantitative storage evidence or qualitative Atlantic prospectivity.
#
# This policy is intentionally explicit so that it can later be changed
# without rebuilding the spatial crosswalk.

regional_storage["storage_accessible"] = (
    regional_storage["has_quantitative_storage_evidence"]
    | regional_storage["has_qualitative_storage_evidence"]
)

regional_storage["storage_accessibility"] = (
    regional_storage["storage_accessible"]
    .astype("int8")
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print("PRELIMINARY GEOCANOE STORAGE ELIGIBILITY")
print("-" * 80)

print(f"Total GeoCANOE regions: {len(regional_storage):,}")

print(
    f"Storage-accessible regions: "
    f"{regional_storage['storage_accessibility'].sum():,}"
)

print(
    f"Storage-inaccessible regions: "
    f"{(regional_storage['storage_accessibility'] == 0).sum():,}"
)

print("\nEvidence classes:")

eligibility_summary = pd.DataFrame(
    {
        "evidence": [
            "NATCARB",
            "BC Storage Atlas",
            "Atlantic COS",
            "Quantitative storage evidence",
            "Qualitative storage evidence",
            "Any accepted storage evidence",
        ],
        "regions": [
            regional_storage["has_natcarb"].sum(),
            regional_storage["has_bc_storage_atlas"].sum(),
            regional_storage["has_atlantic_cos"].sum(),
            regional_storage["has_quantitative_storage_evidence"].sum(),
            regional_storage["has_qualitative_storage_evidence"].sum(),
            regional_storage["storage_accessible"].sum(),
        ],
    }
)

display(eligibility_summary)


print("\nBinary accessibility parameter:")

display(
    regional_storage["storage_accessibility"]
    .value_counts()
    .sort_index()
    .rename_axis("storage_accessibility")
    .reset_index(name="regions")
)

PRELIMINARY GEOCANOE STORAGE ELIGIBILITY
--------------------------------------------------------------------------------
Total GeoCANOE regions: 9,269
Storage-accessible regions: 2,735
Storage-inaccessible regions: 6,534

Evidence classes:


,evidence,regions
0,NATCARB,2579
1,BC Storage Atlas,157
2,Atlantic COS,156
3,Quantitative storage evidence,2579
4,Qualitative storage evidence,156
5,Any accepted storage evidence,2735



Binary accessibility parameter:


,storage_accessibility,regions
0,0,6534
1,1,2735


In [17]:
# ---------------------------------------------------------------------------
# Cell 17 — Map model-ready geological storage accessibility
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Prepare regional accessibility map
# ---------------------------------------------------------------------------

storage_accessibility_map = regional_storage[
    [
        "region",
        "site_id",
        "has_natcarb",
        "has_bc_storage_atlas",
        "has_atlantic_cos",
        "has_quantitative_storage_evidence",
        "has_qualitative_storage_evidence",
        "storage_accessibility",
        "geometry",
    ]
].copy()


# Human-readable fields for interactive inspection.

storage_accessibility_map = storage_accessibility_map.rename(
    columns={
        "region": "GeoCANOE Region",
        "site_id": "Site ID",
        "has_natcarb": "NATCARB",
        "has_bc_storage_atlas": "BC Storage Atlas",
        "has_atlantic_cos": "Atlantic COS",
        "has_quantitative_storage_evidence": "Quantitative Evidence",
        "has_qualitative_storage_evidence": "Qualitative Evidence",
        "storage_accessibility": "Storage Accessibility",
    }
)


# ---------------------------------------------------------------------------
# Split accessible and inaccessible regions
# ---------------------------------------------------------------------------

accessible_regions = storage_accessibility_map[
    storage_accessibility_map["Storage Accessibility"] == 1
].copy()

inaccessible_regions = storage_accessibility_map[
    storage_accessibility_map["Storage Accessibility"] == 0
].copy()


# ---------------------------------------------------------------------------
# Build map layers
# ---------------------------------------------------------------------------

# Inaccessible GeoCANOE regions provide the national model-grid context.
inaccessible_layer = PolygonLayer.from_geopandas(
    inaccessible_regions,
    get_fill_color=[220, 220, 220, 20],
    get_line_color=[130, 130, 130, 70],
    line_width_min_pixels=0.4,
)

# Accessible regions represent A_r = 1.
accessible_layer = PolygonLayer.from_geopandas(
    accessible_regions,
    get_fill_color=[35, 139, 69, 130],
    get_line_color=[20, 90, 50, 180],
    line_width_min_pixels=0.6,
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print("MODEL-READY GEOLOGICAL STORAGE ACCESSIBILITY MAP")
print("-" * 80)

print(f"Total regions: {len(storage_accessibility_map):,}")
print(f"Accessible (A_r = 1): {len(accessible_regions):,}")
print(f"Inaccessible (A_r = 0): {len(inaccessible_regions):,}")


# ---------------------------------------------------------------------------
# Interactive map
# ---------------------------------------------------------------------------

storage_accessibility_view = Map(
    layers=[
        inaccessible_layer,
        accessible_layer,
    ]
)

storage_accessibility_view

MODEL-READY GEOLOGICAL STORAGE ACCESSIBILITY MAP
--------------------------------------------------------------------------------
Total regions: 9,269
Accessible (A_r = 1): 2,735
Inaccessible (A_r = 0): 6,534


c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(
c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(


In [18]:
# ---------------------------------------------------------------------------
# Cell 18 — Build quantitative storage-capacity evidence configuration
# ---------------------------------------------------------------------------

# Configuration 1 (Cell 16):
#
#     storage_accessibility = 1
#
# if a GeoCANOE region contains accepted geological storage evidence,
# regardless of whether a finite storage-capacity estimate is available.
#
# Configuration 2 developed here is stricter:
#
#     capacity_evidence_accessible = 1
#
# only if the region intersects a storage feature associated with at least
# one positive quantitative storage-capacity estimate.
#
# IMPORTANT:
# This cell identifies the PRESENCE of capacity evidence only.
# It does not allocate, aggregate, or sum storage capacity by GeoCANOE region.


# ---------------------------------------------------------------------------
# 1. Define capacity evidence on resolved storage features
# ---------------------------------------------------------------------------

capacity_evidence_features = storage_features_assessed.copy()

capacity_fields = [
    "storage_p10_mt",
    "storage_p50_mt",
    "storage_p90_mt",
    "theoretical_storage_mt",
    "effective_storage_mt",
]


# Individual evidence flags.
#
# Missing values remain "no evidence"; zero-valued estimates are not treated
# as positive capacity evidence.

capacity_evidence_features["has_p10_capacity"] = (
    capacity_evidence_features["storage_p10_mt"].fillna(0) > 0
)

capacity_evidence_features["has_p50_capacity"] = (
    capacity_evidence_features["storage_p50_mt"].fillna(0) > 0
)

capacity_evidence_features["has_p90_capacity"] = (
    capacity_evidence_features["storage_p90_mt"].fillna(0) > 0
)

capacity_evidence_features["has_theoretical_capacity"] = (
    capacity_evidence_features["theoretical_storage_mt"].fillna(0) > 0
)

capacity_evidence_features["has_effective_capacity"] = (
    capacity_evidence_features["effective_storage_mt"].fillna(0) > 0
)


# Any accepted quantitative capacity estimate.
capacity_evidence_features["has_any_capacity_evidence"] = (
    capacity_evidence_features[
        [
            "has_p10_capacity",
            "has_p50_capacity",
            "has_p90_capacity",
            "has_theoretical_capacity",
            "has_effective_capacity",
        ]
    ]
    .any(axis=1)
)


# ---------------------------------------------------------------------------
# 2. QA capacity evidence at the storage-feature level
# ---------------------------------------------------------------------------

print("STORAGE FEATURE CAPACITY EVIDENCE")
print("-" * 80)

feature_capacity_summary = pd.DataFrame(
    {
        "capacity_evidence": [
            "P10",
            "P50",
            "P90",
            "Theoretical",
            "Effective",
            "Any accepted capacity estimate",
        ],
        "storage_features": [
            capacity_evidence_features["has_p10_capacity"].sum(),
            capacity_evidence_features["has_p50_capacity"].sum(),
            capacity_evidence_features["has_p90_capacity"].sum(),
            capacity_evidence_features["has_theoretical_capacity"].sum(),
            capacity_evidence_features["has_effective_capacity"].sum(),
            capacity_evidence_features["has_any_capacity_evidence"].sum(),
        ],
    }
)

display(feature_capacity_summary)


print("\nAny capacity evidence by source:")

display(
    capacity_evidence_features
    .groupby("source_dataset")["has_any_capacity_evidence"]
    .agg(
        storage_features="size",
        features_with_capacity="sum",
    )
    .reset_index()
)


# ---------------------------------------------------------------------------
# 3. Build feature-level capacity-evidence lookup
# ---------------------------------------------------------------------------

capacity_flag_fields = [
    "storage_feature_id",
    "has_p10_capacity",
    "has_p50_capacity",
    "has_p90_capacity",
    "has_theoretical_capacity",
    "has_effective_capacity",
    "has_any_capacity_evidence",
]

capacity_feature_lookup = (
    capacity_evidence_features[capacity_flag_fields]
    .copy()
)


# ---------------------------------------------------------------------------
# 4. Attach capacity evidence to storage × GeoCANOE intersections
# ---------------------------------------------------------------------------

capacity_region_intersections = (
    storage_region_intersections
    .merge(
        capacity_feature_lookup,
        on="storage_feature_id",
        how="left",
        validate="many_to_one",
    )
)


# ---------------------------------------------------------------------------
# 5. Collapse evidence flags to one record per GeoCANOE region
# ---------------------------------------------------------------------------

regional_capacity_evidence = (
    capacity_region_intersections
    .groupby(
        ["region", "site_id"],
        as_index=False,
    )[
        [
            "has_p10_capacity",
            "has_p50_capacity",
            "has_p90_capacity",
            "has_theoretical_capacity",
            "has_effective_capacity",
            "has_any_capacity_evidence",
        ]
    ]
    .any()
)


# ---------------------------------------------------------------------------
# 6. Attach evidence flags to the complete GeoCANOE basemap
# ---------------------------------------------------------------------------

capacity_accessibility = regions_25km[
    [
        "region",
        "site_id",
        "geometry",
    ]
].copy()

capacity_accessibility = capacity_accessibility.merge(
    regional_capacity_evidence,
    on=["region", "site_id"],
    how="left",
    validate="one_to_one",
)


regional_flag_fields = [
    "has_p10_capacity",
    "has_p50_capacity",
    "has_p90_capacity",
    "has_theoretical_capacity",
    "has_effective_capacity",
    "has_any_capacity_evidence",
]

capacity_accessibility[regional_flag_fields] = (
    capacity_accessibility[regional_flag_fields]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)


# Model-ready binary flag for configuration 2.
capacity_accessibility["capacity_evidence_accessibility"] = (
    capacity_accessibility["has_any_capacity_evidence"]
    .astype("int8")
)


# ---------------------------------------------------------------------------
# 7. QA regional capacity-evidence configuration
# ---------------------------------------------------------------------------

print("\nGEOCANOE CAPACITY-EVIDENCE ACCESSIBILITY")
print("-" * 80)

regional_capacity_summary = pd.DataFrame(
    {
        "capacity_evidence": [
            "P10",
            "P50",
            "P90",
            "Theoretical",
            "Effective",
            "Any accepted capacity estimate",
        ],
        "regions": [
            capacity_accessibility["has_p10_capacity"].sum(),
            capacity_accessibility["has_p50_capacity"].sum(),
            capacity_accessibility["has_p90_capacity"].sum(),
            capacity_accessibility["has_theoretical_capacity"].sum(),
            capacity_accessibility["has_effective_capacity"].sum(),
            capacity_accessibility["has_any_capacity_evidence"].sum(),
        ],
    }
)

display(regional_capacity_summary)


print("\nBinary capacity-evidence parameter:")

display(
    capacity_accessibility["capacity_evidence_accessibility"]
    .value_counts()
    .sort_index()
    .rename_axis("capacity_evidence_accessibility")
    .reset_index(name="regions")
)


# ---------------------------------------------------------------------------
# 8. Compare configuration 1 vs configuration 2
# ---------------------------------------------------------------------------

configuration_comparison = (
    regional_storage[
        [
            "region",
            "site_id",
            "storage_accessibility",
        ]
    ]
    .merge(
        capacity_accessibility[
            [
                "region",
                "site_id",
                "capacity_evidence_accessibility",
            ]
        ],
        on=["region", "site_id"],
        how="left",
        validate="one_to_one",
    )
)

print("\nCONFIGURATION COMPARISON")

display(
    configuration_comparison
    .groupby(
        [
            "storage_accessibility",
            "capacity_evidence_accessibility",
        ]
    )
    .size()
    .reset_index(name="regions")
)

STORAGE FEATURE CAPACITY EVIDENCE
--------------------------------------------------------------------------------


,capacity_evidence,storage_features
0,P10,20676
1,P50,20676
2,P90,20676
3,Theoretical,1338
4,Effective,1309
5,Any accepted capacity estimate,21985



Any capacity evidence by source:


,source_dataset,storage_features,features_with_capacity
0,ATLANTIC_COS,4711,0
1,BC_STORAGE_ATLAS,1338,1338
2,NATCARB,29271,20647



GEOCANOE CAPACITY-EVIDENCE ACCESSIBILITY
--------------------------------------------------------------------------------


,capacity_evidence,regions
0,P10,1465
1,P50,1465
2,P90,1465
3,Theoretical,157
4,Effective,136
5,Any accepted capacity estimate,1483



Binary capacity-evidence parameter:


,capacity_evidence_accessibility,regions
0,0,7786
1,1,1483



CONFIGURATION COMPARISON


,storage_accessibility,capacity_evidence_accessibility,regions
0,0,0,6534
1,1,0,1252
2,1,1,1483


In [19]:
# ---------------------------------------------------------------------------
# Cell 19 — Map quantitative storage-capacity evidence accessibility
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Prepare model-ready capacity-evidence map
# ---------------------------------------------------------------------------

capacity_evidence_map = capacity_accessibility[
    [
        "region",
        "site_id",
        "has_p10_capacity",
        "has_p50_capacity",
        "has_p90_capacity",
        "has_theoretical_capacity",
        "has_effective_capacity",
        "has_any_capacity_evidence",
        "capacity_evidence_accessibility",
        "geometry",
    ]
].copy()

capacity_evidence_map = capacity_evidence_map.rename(
    columns={
        "region": "GeoCANOE Region",
        "site_id": "Site ID",
        "has_p10_capacity": "P10 Evidence",
        "has_p50_capacity": "P50 Evidence",
        "has_p90_capacity": "P90 Evidence",
        "has_theoretical_capacity": "Theoretical Evidence",
        "has_effective_capacity": "Effective Evidence",
        "has_any_capacity_evidence": "Any Capacity Evidence",
        "capacity_evidence_accessibility": "Capacity Evidence Accessibility",
    }
)


# ---------------------------------------------------------------------------
# Split eligible and ineligible regions
# ---------------------------------------------------------------------------

capacity_eligible_regions = capacity_evidence_map[
    capacity_evidence_map["Capacity Evidence Accessibility"] == 1
].copy()

capacity_ineligible_regions = capacity_evidence_map[
    capacity_evidence_map["Capacity Evidence Accessibility"] == 0
].copy()


# ---------------------------------------------------------------------------
# Build map layers
# ---------------------------------------------------------------------------

capacity_ineligible_layer = PolygonLayer.from_geopandas(
    capacity_ineligible_regions,
    get_fill_color=[220, 220, 220, 20],
    get_line_color=[130, 130, 130, 70],
    line_width_min_pixels=0.4,
)

capacity_eligible_layer = PolygonLayer.from_geopandas(
    capacity_eligible_regions,
    get_fill_color=[35, 139, 69, 130],
    get_line_color=[20, 90, 50, 180],
    line_width_min_pixels=0.6,
)


# ---------------------------------------------------------------------------
# QA
# ---------------------------------------------------------------------------

print("MODEL-READY CAPACITY-EVIDENCE ACCESSIBILITY MAP")
print("-" * 80)

print(f"Total regions: {len(capacity_evidence_map):,}")
print(f"Capacity evidence = 1: {len(capacity_eligible_regions):,}")
print(f"Capacity evidence = 0: {len(capacity_ineligible_regions):,}")


# ---------------------------------------------------------------------------
# Interactive map
# ---------------------------------------------------------------------------

capacity_evidence_view = Map(
    layers=[
        capacity_ineligible_layer,
        capacity_eligible_layer,
    ]
)

capacity_evidence_view

MODEL-READY CAPACITY-EVIDENCE ACCESSIBILITY MAP
--------------------------------------------------------------------------------
Total regions: 9,269
Capacity evidence = 1: 1,483
Capacity evidence = 0: 7,786


c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(
c:\Users\aviga\Research\repos\temoa-upstream\temoa_geospace\.venv\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:116: UserWarning: Input being reprojected to EPSG:4326 CRS.
Lonboard is only able to render data in EPSG:4326 projection.
  warnings.warn(


In [20]:
# ---------------------------------------------------------------------------
# Diagnostic — All storage features intersecting GeoCANOE region R0
# ---------------------------------------------------------------------------

r0_features = (
    capacity_region_intersections.loc[
        capacity_region_intersections["region"] == "R0"
    ]
    [
        [
            "region",
            "site_id",
            "storage_feature_id",
            "storage_unit_id",
            "source_dataset",
            "storage_type",
            "representation",
            "region_overlap_fraction",
            "feature_overlap_fraction",
            "has_p10_capacity",
            "has_p50_capacity",
            "has_p90_capacity",
            "has_theoretical_capacity",
            "has_effective_capacity",
            "has_any_capacity_evidence",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "has_any_capacity_evidence",
            "source_dataset",
            "storage_feature_id",
        ],
        ascending=[False, True, True],
    )
)

print(f"Storage features intersecting R0: {len(r0_features):,}")
print(
    f"Positive-capacity features: "
    f"{r0_features['has_any_capacity_evidence'].sum():,}"
)

display(r0_features)

r0_capacity_values = (
    capacity_region_intersections.loc[
        capacity_region_intersections["region"] == "R0",
        [
            "region",
            "site_id",
            "storage_feature_id",
            "storage_unit_id",
            "source_dataset",
            "storage_type",
        ],
    ]
    .drop_duplicates()
    .merge(
        storage_features_assessed[
            [
                "storage_feature_id",
                "storage_p10_mt",
                "storage_p50_mt",
                "storage_p90_mt",
                "theoretical_storage_mt",
                "effective_storage_mt",
            ]
        ],
        on="storage_feature_id",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        ["source_dataset", "storage_feature_id"]
    )
)

display(r0_capacity_values)

capacity_columns = [
    "storage_p10_mt",
    "storage_p50_mt",
    "storage_p90_mt",
    "theoretical_storage_mt",
    "effective_storage_mt",
]

print("Maximum capacity values found anywhere in R0:")
display(r0_capacity_values[capacity_columns].max())

print("\nFeatures with ANY positive capacity value:")
display(
    r0_capacity_values.loc[
        r0_capacity_values[capacity_columns]
        .fillna(0)
        .gt(0)
        .any(axis=1)
    ]
)

Storage features intersecting R0: 12
Positive-capacity features: 2


,region,site_id,storage_feature_id,storage_unit_id,source_dataset,storage_type,representation,region_overlap_fraction,feature_overlap_fraction,has_p10_capacity,has_p50_capacity,has_p90_capacity,has_theoretical_capacity,has_effective_capacity,has_any_capacity_evidence
5,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941200,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,0.004242,0.024925,True,True,True,False,False,True
8,R0,R0,NATCARB_SALINE_natcarb:saline_cell:946065,NAT_SAL_8dbf6c8c8487,NATCARB,saline_aquifer,resource_grid_cell,0.001676,0.009839,True,True,True,False,False,True
0,R0,R0,NATCARB_SALINE_natcarb:saline_cell:933344,NAT_SAL_a39aa680ed6b,NATCARB,saline_aquifer,resource_grid_cell,0.004242,0.024925,False,False,False,False,False,False
1,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936129,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,0.001676,0.009839,False,False,False,False,False,False
2,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936130,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,0.004242,0.024925,False,False,False,False,False,False
3,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936149,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,0.089193,0.523917,False,False,False,False,False,False
4,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941199,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,0.001676,0.009839,False,False,False,False,False,False
6,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941255,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,0.089193,0.523917,False,False,False,False,False,False
7,R0,R0,NATCARB_SALINE_natcarb:saline_cell:945964,NAT_SAL_fb32f9a4a3a7,NATCARB,saline_aquifer,resource_grid_cell,0.004242,0.024925,False,False,False,False,False,False
9,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947581,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,resource_grid_cell,0.001676,0.009839,False,False,False,False,False,False


,region,site_id,storage_feature_id,storage_unit_id,source_dataset,storage_type,storage_p10_mt,storage_p50_mt,storage_p90_mt,theoretical_storage_mt,effective_storage_mt
0,R0,R0,NATCARB_SALINE_natcarb:saline_cell:933344,NAT_SAL_a39aa680ed6b,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN
1,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936129,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN
2,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936130,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN
3,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936149,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN
4,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941199,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN
5,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941200,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,0.436317,0.436317,0.436317,NaN,NaN
6,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941255,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN
7,R0,R0,NATCARB_SALINE_natcarb:saline_cell:945964,NAT_SAL_fb32f9a4a3a7,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN
8,R0,R0,NATCARB_SALINE_natcarb:saline_cell:946065,NAT_SAL_8dbf6c8c8487,NATCARB,saline_aquifer,0.164654,0.164654,0.164654,NaN,NaN
9,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947581,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,0.000000,0.000000,0.000000,NaN,NaN


Maximum capacity values found anywhere in R0:


storage_p10_mt            0.436317
storage_p50_mt            0.436317
storage_p90_mt            0.436317
theoretical_storage_mt         NaN
effective_storage_mt           NaN
dtype: float64


Features with ANY positive capacity value:


,region,site_id,storage_feature_id,storage_unit_id,source_dataset,storage_type,storage_p10_mt,storage_p50_mt,storage_p90_mt,theoretical_storage_mt,effective_storage_mt
5,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941200,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,0.436317,0.436317,0.436317,NaN,NaN
8,R0,R0,NATCARB_SALINE_natcarb:saline_cell:946065,NAT_SAL_8dbf6c8c8487,NATCARB,saline_aquifer,0.164654,0.164654,0.164654,NaN,NaN


In [21]:
# ---------------------------------------------------------------------------
# Diagnostic — Capacity-evidence spatial-overlap threshold sensitivity
# ---------------------------------------------------------------------------

# Test progressively stricter minimum fractions of the GeoCANOE region that
# must be covered by a POSITIVE-CAPACITY storage feature.
#
# This does not modify capacity_accessibility or the model-ready configuration.
# It is a sensitivity test only.

overlap_thresholds = [
    0.000,
    0.001,
    0.005,
    0.010,
    0.025,
    0.050,
    0.100,
    0.250,
]


# ---------------------------------------------------------------------------
# 1. Retain only intersections carrying positive capacity evidence
# ---------------------------------------------------------------------------

positive_capacity_intersections = (
    capacity_region_intersections.loc[
        capacity_region_intersections["has_any_capacity_evidence"]
    ]
    .copy()
)

print("POSITIVE-CAPACITY SPATIAL INTERSECTIONS")
print("-" * 80)

print(
    f"Feature × region intersections: "
    f"{len(positive_capacity_intersections):,}"
)

print(
    f"Unique capacity-bearing features: "
    f"{positive_capacity_intersections['storage_feature_id'].nunique():,}"
)

print(
    f"GeoCANOE regions at zero threshold: "
    f"{positive_capacity_intersections['region'].nunique():,}"
)


# ---------------------------------------------------------------------------
# 2. Test regional-overlap thresholds
# ---------------------------------------------------------------------------

threshold_records = []

for threshold in overlap_thresholds:

    qualifying = positive_capacity_intersections.loc[
        positive_capacity_intersections["region_overlap_fraction"] >= threshold
    ]

    qualifying_regions = qualifying["region"].nunique()

    threshold_records.append(
        {
            "minimum_region_overlap_fraction": threshold,
            "minimum_region_overlap_percent": threshold * 100,
            "qualifying_regions": qualifying_regions,
            "regions_removed": (
                positive_capacity_intersections["region"].nunique()
                - qualifying_regions
            ),
            "percent_of_original_regions_retained": (
                qualifying_regions
                / positive_capacity_intersections["region"].nunique()
                * 100
            ),
        }
    )

capacity_threshold_sensitivity = pd.DataFrame(threshold_records)

print("\nREGIONAL CAPACITY-EVIDENCE THRESHOLD SENSITIVITY")

display(capacity_threshold_sensitivity)


# ---------------------------------------------------------------------------
# 3. Source-specific sensitivity
# ---------------------------------------------------------------------------

source_threshold_records = []

for threshold in overlap_thresholds:

    qualifying = positive_capacity_intersections.loc[
        positive_capacity_intersections["region_overlap_fraction"] >= threshold
    ]

    for source_dataset, source_data in qualifying.groupby("source_dataset"):

        source_threshold_records.append(
            {
                "minimum_region_overlap_fraction": threshold,
                "minimum_region_overlap_percent": threshold * 100,
                "source_dataset": source_dataset,
                "qualifying_regions": source_data["region"].nunique(),
            }
        )

capacity_threshold_by_source = pd.DataFrame(source_threshold_records)

print("\nTHRESHOLD SENSITIVITY BY SOURCE")

display(
    capacity_threshold_by_source.pivot(
        index="minimum_region_overlap_percent",
        columns="source_dataset",
        values="qualifying_regions",
    )
)


# ---------------------------------------------------------------------------
# 4. Inspect the threshold at which R0 loses eligibility
# ---------------------------------------------------------------------------

r0_threshold_test = pd.DataFrame(
    {
        "minimum_region_overlap_percent": [
            threshold * 100
            for threshold in overlap_thresholds
        ],
        "R0_capacity_eligible": [
            (
                (
                    positive_capacity_intersections["region"] == "R0"
                )
                &
                (
                    positive_capacity_intersections[
                        "region_overlap_fraction"
                    ] >= threshold
                )
            ).any()
            for threshold in overlap_thresholds
        ],
    }
)

print("\nR0 THRESHOLD TEST")

display(r0_threshold_test)

POSITIVE-CAPACITY SPATIAL INTERSECTIONS
--------------------------------------------------------------------------------
Feature × region intersections: 43,348
Unique capacity-bearing features: 21,619
GeoCANOE regions at zero threshold: 1,483

REGIONAL CAPACITY-EVIDENCE THRESHOLD SENSITIVITY


,minimum_region_overlap_fraction,minimum_region_overlap_percent,qualifying_regions,regions_removed,percent_of_original_regions_retained
0,0.000,0.0,1483,0,100.000000
1,0.001,0.1,1478,5,99.662846
2,0.005,0.5,1466,17,98.853675
3,0.010,1.0,1462,21,98.583951
4,0.025,2.5,1450,33,97.774781
5,0.050,5.0,1432,51,96.561025
6,0.100,10.0,1404,79,94.672960
7,0.250,25.0,147,1336,9.912340



THRESHOLD SENSITIVITY BY SOURCE


source_dataset,BC_STORAGE_ATLAS,NATCARB
minimum_region_overlap_percent,,
0.0,157,1412
0.1,154,1408
0.5,152,1397
1.0,150,1392
2.5,146,1383
5.0,137,1368
10.0,127,1345
25.0,119,28



R0 THRESHOLD TEST


,minimum_region_overlap_percent,R0_capacity_eligible
0,0.0,True
1,0.1,True
2,0.5,False
3,1.0,False
4,2.5,False
5,5.0,False
6,10.0,False
7,25.0,False


In [22]:
# ---------------------------------------------------------------------------
# Diagnostic — Unioned positive-capacity coverage by GeoCANOE region
# ---------------------------------------------------------------------------

# Goal:
#
# For each GeoCANOE region r, calculate:
#
#     capacity_evidence_coverage_fraction =
#
#         area(
#             union of all positive-capacity storage geometry within r
#         )
#         ------------------------------------------------------------
#                         area of region r
#
# This avoids:
#   1. treating tiny individual polygon intersections as independently
#      representative of the whole GeoCANOE region;
#   2. double-counting overlapping storage polygons;
#   3. making the threshold dependent on source polygon tessellation.


# ---------------------------------------------------------------------------
# 1. Select positive-capacity intersection geometries
# ---------------------------------------------------------------------------

positive_capacity_geometry = (
    capacity_region_intersections.loc[
        capacity_region_intersections["has_any_capacity_evidence"],
        ["region", "site_id", "geometry"],
    ]
    .copy()
)

print("POSITIVE-CAPACITY INTERSECTION GEOMETRY")
print("-" * 80)

print(
    f"Feature × region intersections: "
    f"{len(positive_capacity_geometry):,}"
)

print(
    f"GeoCANOE regions represented: "
    f"{positive_capacity_geometry['region'].nunique():,}"
)


# ---------------------------------------------------------------------------
# 2. Union positive-capacity geometry within each GeoCANOE region
# ---------------------------------------------------------------------------

positive_capacity_union = (
    positive_capacity_geometry
    .dissolve(
        by=["region", "site_id"],
        as_index=False,
    )
)

positive_capacity_union["capacity_evidence_area_m2"] = (
    positive_capacity_union.geometry.area
)


# ---------------------------------------------------------------------------
# 3. Attach full GeoCANOE region area
# ---------------------------------------------------------------------------

region_area_lookup = regions_25km[
    [
        "region",
        "site_id",
        "geometry",
    ]
].copy()

region_area_lookup["region_area_m2"] = (
    region_area_lookup.geometry.area
)

region_area_lookup = region_area_lookup[
    [
        "region",
        "site_id",
        "region_area_m2",
    ]
]


positive_capacity_union = positive_capacity_union.merge(
    region_area_lookup,
    on=["region", "site_id"],
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------------------------
# 4. Calculate unioned capacity-evidence coverage
# ---------------------------------------------------------------------------

positive_capacity_union["capacity_evidence_coverage_fraction"] = (
    positive_capacity_union["capacity_evidence_area_m2"]
    / positive_capacity_union["region_area_m2"]
)

positive_capacity_union["capacity_evidence_coverage_percent"] = (
    positive_capacity_union["capacity_evidence_coverage_fraction"] * 100
)


# ---------------------------------------------------------------------------
# 5. Basic QA
# ---------------------------------------------------------------------------

print("\nUNIONED CAPACITY-EVIDENCE COVERAGE")
print("-" * 80)

display(
    positive_capacity_union[
        "capacity_evidence_coverage_percent"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)


# ---------------------------------------------------------------------------
# 6. Threshold sensitivity using UNIONED regional coverage
# ---------------------------------------------------------------------------

union_overlap_thresholds = [
    0.000,
    0.001,
    0.005,
    0.010,
    0.025,
    0.050,
    0.100,
    0.250,
    0.500,
]

union_threshold_records = []

total_capacity_regions = len(positive_capacity_union)

for threshold in union_overlap_thresholds:

    qualifying_regions = (
        positive_capacity_union[
            "capacity_evidence_coverage_fraction"
        ]
        .ge(threshold)
        .sum()
    )

    union_threshold_records.append(
        {
            "minimum_capacity_coverage_fraction": threshold,
            "minimum_capacity_coverage_percent": threshold * 100,
            "qualifying_regions": qualifying_regions,
            "regions_removed": (
                total_capacity_regions - qualifying_regions
            ),
            "percent_of_original_regions_retained": (
                qualifying_regions
                / total_capacity_regions
                * 100
            ),
        }
    )

capacity_union_threshold_sensitivity = pd.DataFrame(
    union_threshold_records
)

print("\nUNIONED REGIONAL COVERAGE THRESHOLD SENSITIVITY")

display(capacity_union_threshold_sensitivity)


# ---------------------------------------------------------------------------
# 7. Inspect R0
# ---------------------------------------------------------------------------

print("\nR0 UNIONED POSITIVE-CAPACITY COVERAGE")

display(
    positive_capacity_union.loc[
        positive_capacity_union["region"] == "R0",
        [
            "region",
            "site_id",
            "capacity_evidence_area_m2",
            "region_area_m2",
            "capacity_evidence_coverage_fraction",
            "capacity_evidence_coverage_percent",
        ],
    ]
)

POSITIVE-CAPACITY INTERSECTION GEOMETRY
--------------------------------------------------------------------------------
Feature × region intersections: 43,348
GeoCANOE regions represented: 1,483

UNIONED CAPACITY-EVIDENCE COVERAGE
--------------------------------------------------------------------------------


count    1483.000000
mean       90.530671
std        24.999180
min         0.012589
1%          0.503226
5%         16.295917
10%        52.745583
25%       100.000000
50%       100.000000
75%       100.000000
90%       100.000000
95%       100.000000
99%       100.000000
max       100.000000
Name: capacity_evidence_coverage_percent, dtype: float64


UNIONED REGIONAL COVERAGE THRESHOLD SENSITIVITY


,minimum_capacity_coverage_fraction,minimum_capacity_coverage_percent,qualifying_regions,regions_removed,percent_of_original_regions_retained
0,0.000,0.0,1483,0,100.000000
1,0.001,0.1,1478,5,99.662846
2,0.005,0.5,1468,15,98.988537
3,0.010,1.0,1463,20,98.651382
4,0.025,2.5,1452,31,97.909643
5,0.050,5.0,1443,40,97.302765
6,0.100,10.0,1424,59,96.021578
7,0.250,25.0,1389,94,93.661497
8,0.500,50.0,1340,143,90.357384



R0 UNIONED POSITIVE-CAPACITY COVERAGE


,region,site_id,capacity_evidence_area_m2,region_area_m2,capacity_evidence_coverage_fraction,capacity_evidence_coverage_percent
0,R0,R0,3.699152e+06,625000000.0,0.005919,0.591864


In [24]:
# ---------------------------------------------------------------------------
# Build integrated regional CO2 storage evidence table
# ---------------------------------------------------------------------------

# This is the GeoCANOE integrated-Silver regional product.
# Scenario-specific eligibility thresholds are deliberately NOT applied here.


# ---------------------------------------------------------------------------
# 1. Start from complete GeoCANOE basemap
# ---------------------------------------------------------------------------

regional_storage_evidence = regions_25km[
    ["region", "site_id", "geometry"]
].copy()


# ---------------------------------------------------------------------------
# 2. Geological / source evidence
# ---------------------------------------------------------------------------

geological_fields = [
    "region",
    "site_id",
    "atlantic_coverage_fraction",
    "bc_coverage_fraction",
    "natcarb_coverage_fraction",
    "has_natcarb",
    "has_bc_storage_atlas",
    "has_atlantic_cos",
    "has_quantitative_storage_evidence",
    "has_qualitative_storage_evidence",
    "storage_accessible",
    "storage_accessibility",
]

regional_storage_evidence = regional_storage_evidence.merge(
    regional_storage[geological_fields],
    on=["region", "site_id"],
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------------------------
# 3. Capacity-assessment evidence
# ---------------------------------------------------------------------------

capacity_flag_fields = [
    "has_p10_capacity",
    "has_p50_capacity",
    "has_p90_capacity",
    "has_theoretical_capacity",
    "has_effective_capacity",
    "has_any_capacity_evidence",
]

regional_storage_evidence = regional_storage_evidence.merge(
    capacity_accessibility[
        ["region", "site_id"] + capacity_flag_fields
    ],
    on=["region", "site_id"],
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------------------------
# 4. Unioned positive-capacity spatial coverage
# ---------------------------------------------------------------------------

capacity_coverage_fields = [
    "region",
    "site_id",
    "capacity_evidence_area_m2",
    "capacity_evidence_coverage_fraction",
    "capacity_evidence_coverage_percent",
]

regional_storage_evidence = regional_storage_evidence.merge(
    positive_capacity_union[capacity_coverage_fields],
    on=["region", "site_id"],
    how="left",
    validate="one_to_one",
)


# ---------------------------------------------------------------------------
# 5. Normalize missing evidence
# ---------------------------------------------------------------------------

source_flag_fields = [
    "has_natcarb",
    "has_bc_storage_atlas",
    "has_atlantic_cos",
    "has_quantitative_storage_evidence",
    "has_qualitative_storage_evidence",
]

boolean_fields = source_flag_fields + capacity_flag_fields

regional_storage_evidence[boolean_fields] = (
    regional_storage_evidence[boolean_fields]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)

regional_storage_evidence["storage_accessible"] = (
    regional_storage_evidence["storage_accessible"]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)

regional_storage_evidence["storage_accessibility"] = (
    regional_storage_evidence["storage_accessibility"]
    .fillna(0)
    .astype("int8")
)

coverage_fields = [
    "atlantic_coverage_fraction",
    "bc_coverage_fraction",
    "natcarb_coverage_fraction",
    "capacity_evidence_area_m2",
    "capacity_evidence_coverage_fraction",
    "capacity_evidence_coverage_percent",
]

regional_storage_evidence[coverage_fields] = (
    regional_storage_evidence[coverage_fields]
    .fillna(0.0)
)


# ---------------------------------------------------------------------------
# 6. Normalize floating-point spatial precision
# ---------------------------------------------------------------------------

# Overlay / union operations produced a maximum observed excess above 1.0
# of 4.44e-16. This is floating-point noise rather than spatial over-coverage.

regional_storage_evidence[
    "capacity_evidence_coverage_fraction"
] = (
    regional_storage_evidence[
        "capacity_evidence_coverage_fraction"
    ]
    .clip(lower=0.0, upper=1.0)
)

regional_storage_evidence[
    "capacity_evidence_coverage_percent"
] = (
    regional_storage_evidence[
        "capacity_evidence_coverage_fraction"
    ] * 100
)


# ---------------------------------------------------------------------------
# 7. QA assertions
# ---------------------------------------------------------------------------

assert len(regional_storage_evidence) == len(regions_25km)
assert regional_storage_evidence["region"].is_unique

assert (
    regional_storage_evidence[
        "capacity_evidence_coverage_fraction"
    ]
    .between(0, 1)
    .all()
)

assert (
    regional_storage_evidence["has_any_capacity_evidence"]
    ==
    regional_storage_evidence[
        [
            "has_p10_capacity",
            "has_p50_capacity",
            "has_p90_capacity",
            "has_theoretical_capacity",
            "has_effective_capacity",
        ]
    ].any(axis=1)
).all()

assert (
    regional_storage_evidence.loc[
        regional_storage_evidence["has_any_capacity_evidence"],
        "storage_accessible",
    ]
).all()


# ---------------------------------------------------------------------------
# 8. Validation summary
# ---------------------------------------------------------------------------

print("REGIONAL CO2 STORAGE EVIDENCE")
print("-" * 80)

print(f"Total GeoCANOE regions: {len(regional_storage_evidence):,}")

print(
    "Geologically accessible: "
    f"{regional_storage_evidence['storage_accessibility'].sum():,}"
)

print(
    "Quantitative geological evidence: "
    f"{regional_storage_evidence['has_quantitative_storage_evidence'].sum():,}"
)

print(
    "Qualitative geological evidence: "
    f"{regional_storage_evidence['has_qualitative_storage_evidence'].sum():,}"
)

print(
    "Positive capacity evidence: "
    f"{regional_storage_evidence['has_any_capacity_evidence'].sum():,}"
)


print("\nSOURCE PRESENCE")
print("-" * 80)

for field in [
    "has_natcarb",
    "has_bc_storage_atlas",
    "has_atlantic_cos",
]:
    print(
        f"{field}: "
        f"{regional_storage_evidence[field].sum():,}"
    )


print("\nCAPACITY EVIDENCE")
print("-" * 80)

for field in capacity_flag_fields:
    print(
        f"{field}: "
        f"{regional_storage_evidence[field].sum():,}"
    )


print("\nPOSITIVE-CAPACITY COVERAGE (%)")
print("-" * 80)

display(
    regional_storage_evidence.loc[
        regional_storage_evidence["has_any_capacity_evidence"],
        "capacity_evidence_coverage_percent",
    ].describe()
)


# ---------------------------------------------------------------------------
# 9. Preview
# ---------------------------------------------------------------------------

display_columns = [
    "region",
    "site_id",
    "has_natcarb",
    "has_bc_storage_atlas",
    "has_atlantic_cos",
    "has_quantitative_storage_evidence",
    "has_qualitative_storage_evidence",
    "storage_accessibility",
    "has_any_capacity_evidence",
    "has_p10_capacity",
    "has_p50_capacity",
    "has_p90_capacity",
    "has_theoretical_capacity",
    "has_effective_capacity",
    "capacity_evidence_coverage_fraction",
]

display(
    regional_storage_evidence[
        display_columns
    ].head(20)
)

REGIONAL CO2 STORAGE EVIDENCE
--------------------------------------------------------------------------------
Total GeoCANOE regions: 9,269
Geologically accessible: 2,735
Quantitative geological evidence: 2,579
Qualitative geological evidence: 156
Positive capacity evidence: 1,483

SOURCE PRESENCE
--------------------------------------------------------------------------------
has_natcarb: 2,579
has_bc_storage_atlas: 157
has_atlantic_cos: 156

CAPACITY EVIDENCE
--------------------------------------------------------------------------------
has_p10_capacity: 1,465
has_p50_capacity: 1,465
has_p90_capacity: 1,465
has_theoretical_capacity: 157
has_effective_capacity: 136
has_any_capacity_evidence: 1,483

POSITIVE-CAPACITY COVERAGE (%)
--------------------------------------------------------------------------------


count    1483.000000
mean       90.530671
std        24.999180
min         0.012589
25%       100.000000
50%       100.000000
75%       100.000000
max       100.000000
Name: capacity_evidence_coverage_percent, dtype: float64

,region,site_id,has_natcarb,has_bc_storage_atlas,has_atlantic_cos,has_quantitative_storage_evidence,has_qualitative_storage_evidence,storage_accessibility,has_any_capacity_evidence,has_p10_capacity,has_p50_capacity,has_p90_capacity,has_theoretical_capacity,has_effective_capacity,capacity_evidence_coverage_fraction
0,R0,R0,True,False,False,True,False,1,True,True,True,True,False,False,0.005919
1,R1,R1,False,False,False,False,False,0,False,False,False,False,False,False,0.000000
2,R2,R2,False,False,False,False,False,0,False,False,False,False,False,False,0.000000
3,R3,R3,False,False,False,False,False,0,False,False,False,False,False,False,0.000000
4,R4,R4,True,False,False,True,False,1,True,True,True,True,False,False,0.104534
5,R5,R5,False,False,False,False,False,0,False,False,False,False,False,False,0.000000
6,R6,R6,False,False,False,False,False,0,False,False,False,False,False,False,0.000000
7,R7,R7,False,False,False,False,False,0,False,False,False,False,False,False,0.000000
8,R8,R8,False,False,False,False,False,0,False,False,False,False,False,False,0.000000
9,R9,R9,False,False,False,False,False,0,False,False,False,False,False,False,0.000000


In [26]:
# ---------------------------------------------------------------------------
# Cell 21 — Build canonical CO2 storage feature → GeoCANOE region crosswalk
# ---------------------------------------------------------------------------

# Purpose
# -------
# Preserve the many-to-many spatial relationship between the external
# geological-storage database and GeoCANOE regions.
#
# This is a spatial topology / lineage product.
#
# It deliberately does NOT:
#   - allocate storage capacity to regions;
#   - sum capacity across features;
#   - reconcile potentially duplicated geological resources across sources;
#   - apply scenario-specific eligibility thresholds.
#
# Those operations belong downstream.


# ---------------------------------------------------------------------------
# 1. Select canonical crosswalk fields
# ---------------------------------------------------------------------------

crosswalk_fields = [
    "region",
    "site_id",
    "storage_feature_id",
    "storage_unit_id",
    "source_dataset",
    "source_layer",
    "storage_type",
    "representation",
    "assessment_type",
    "data_class",
    "capacity_data",
    "injectivity_status",
    "region_area_m2",
    "storage_feature_area_m2",
    "intersection_area_m2",
    "region_overlap_fraction",
    "feature_overlap_fraction",
    "geometry",
]

storage_region_crosswalk = (
    storage_region_intersections[crosswalk_fields]
    .copy()
)


# ---------------------------------------------------------------------------
# 2. Normalize floating-point spatial precision
# ---------------------------------------------------------------------------

# Overlay operations can produce infinitesimal floating-point excursions
# outside the theoretical [0, 1] range.

for field in [
    "region_overlap_fraction",
    "feature_overlap_fraction",
]:
    storage_region_crosswalk[field] = (
        storage_region_crosswalk[field]
        .clip(lower=0.0, upper=1.0)
    )


# ---------------------------------------------------------------------------
# 3. Canonical ordering
# ---------------------------------------------------------------------------

storage_region_crosswalk = (
    storage_region_crosswalk
    .sort_values(
        [
            "region",
            "source_dataset",
            "storage_unit_id",
            "storage_feature_id",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# 4. QA assertions
# ---------------------------------------------------------------------------

assert storage_region_crosswalk.crs == regions_25km.crs

assert (
    storage_region_crosswalk["intersection_area_m2"] >= 0
).all()

assert (
    storage_region_crosswalk["region_overlap_fraction"]
    .between(0, 1)
    .all()
)

assert (
    storage_region_crosswalk["feature_overlap_fraction"]
    .between(0, 1)
    .all()
)

assert (
    storage_region_crosswalk["region"]
    .isin(regions_25km["region"])
    .all()
)

assert (
    storage_region_crosswalk["storage_feature_id"]
    .isin(storage_features_projected["storage_feature_id"])
    .all()
)


# ---------------------------------------------------------------------------
# 5. Crosswalk uniqueness QA
# ---------------------------------------------------------------------------

# Each record should represent one storage-feature × GeoCANOE-region
# intersection. A feature may intersect many regions and a region may
# intersect many features.

duplicate_pairs = storage_region_crosswalk.duplicated(
    subset=[
        "region",
        "site_id",
        "storage_feature_id",
    ],
    keep=False,
)

print("CO2 STORAGE → GEOCANOE REGION CROSSWALK")
print("-" * 80)

print(
    f"Crosswalk records: "
    f"{len(storage_region_crosswalk):,}"
)

print(
    f"GeoCANOE regions represented: "
    f"{storage_region_crosswalk['region'].nunique():,}"
)

print(
    f"Storage features represented: "
    f"{storage_region_crosswalk['storage_feature_id'].nunique():,}"
)

print(
    f"Logical storage units represented: "
    f"{storage_region_crosswalk['storage_unit_id'].nunique():,}"
)

print(
    f"Duplicate region × feature pairs: "
    f"{duplicate_pairs.sum():,}"
)


# ---------------------------------------------------------------------------
# 6. Source / storage-type QA
# ---------------------------------------------------------------------------

print("\nCROSSWALK RECORDS BY SOURCE")
print("-" * 80)

print(
    storage_region_crosswalk["source_dataset"]
    .value_counts(dropna=False)
)


print("\nSTORAGE FEATURES BY SOURCE")
print("-" * 80)

print(
    storage_region_crosswalk
    .groupby("source_dataset")["storage_feature_id"]
    .nunique()
    .sort_values(ascending=False)
)


print("\nLOGICAL STORAGE UNITS BY SOURCE")
print("-" * 80)

print(
    storage_region_crosswalk
    .groupby("source_dataset")["storage_unit_id"]
    .nunique()
    .sort_values(ascending=False)
)


print("\nSTORAGE TYPES")
print("-" * 80)

print(
    storage_region_crosswalk["storage_type"]
    .value_counts(dropna=False)
)


# ---------------------------------------------------------------------------
# 7. Spatial overlap QA
# ---------------------------------------------------------------------------

print("\nSPATIAL OVERLAP FRACTIONS")
print("-" * 80)

display(
    storage_region_crosswalk[
        [
            "region_overlap_fraction",
            "feature_overlap_fraction",
        ]
    ].describe()
)


# ---------------------------------------------------------------------------
# 8. Preview
# ---------------------------------------------------------------------------

display(
    storage_region_crosswalk[
        [
            "region",
            "site_id",
            "storage_feature_id",
            "storage_unit_id",
            "source_dataset",
            "storage_type",
            "representation",
            "intersection_area_m2",
            "region_overlap_fraction",
            "feature_overlap_fraction",
        ]
    ].head(20)
)

CO2 STORAGE → GEOCANOE REGION CROSSWALK
--------------------------------------------------------------------------------
Crosswalk records: 61,748
GeoCANOE regions represented: 2,735
Storage features represented: 30,637
Logical storage units represented: 2,819
Duplicate region × feature pairs: 0

CROSSWALK RECORDS BY SOURCE
--------------------------------------------------------------------------------
source_dataset
NATCARB             57875
BC_STORAGE_ATLAS     2332
ATLANTIC_COS         1541
Name: count, dtype: int64

STORAGE FEATURES BY SOURCE
--------------------------------------------------------------------------------
source_dataset
NATCARB             28587
BC_STORAGE_ATLAS     1338
ATLANTIC_COS          712
Name: storage_feature_id, dtype: int64

LOGICAL STORAGE UNITS BY SOURCE
--------------------------------------------------------------------------------
source_dataset
NATCARB             1547
BC_STORAGE_ATLAS    1263
ATLANTIC_COS           9
Name: storage_unit_id, dtype:

,region_overlap_fraction,feature_overlap_fraction
count,6.174800e+04,6.174800e+04
mean,7.592483e-02,4.829285e-01
std,7.742266e-02,3.652884e-01
min,1.792617e-09,1.156370e-08
25%,1.736208e-02,1.311246e-01
50%,6.157542e-02,4.299933e-01
75%,1.291884e-01,8.643069e-01
max,1.000000e+00,1.000000e+00


,region,site_id,storage_feature_id,storage_unit_id,source_dataset,storage_type,representation,intersection_area_m2,region_overlap_fraction,feature_overlap_fraction
0,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941199,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,1.047661e+06,0.001676,0.009839
1,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941200,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,2.651491e+06,0.004242,0.024925
2,R0,R0,NATCARB_SALINE_natcarb:saline_cell:941255,NAT_SAL_311f7d63fa70,NATCARB,saline_aquifer,resource_grid_cell,5.574553e+07,0.089193,0.523917
3,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936129,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,1.047661e+06,0.001676,0.009839
4,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936130,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,2.651491e+06,0.004242,0.024925
5,R0,R0,NATCARB_SALINE_natcarb:saline_cell:936149,NAT_SAL_5361b04c6542,NATCARB,saline_aquifer,resource_grid_cell,5.574553e+07,0.089193,0.523917
6,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947581,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,resource_grid_cell,1.047661e+06,0.001676,0.009839
7,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947582,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,resource_grid_cell,2.651491e+06,0.004242,0.024925
8,R0,R0,NATCARB_SALINE_natcarb:saline_cell:947584,NAT_SAL_7d4667380b63,NATCARB,saline_aquifer,resource_grid_cell,5.574553e+07,0.089193,0.523917
9,R0,R0,NATCARB_SALINE_natcarb:saline_cell:946065,NAT_SAL_8dbf6c8c8487,NATCARB,saline_aquifer,resource_grid_cell,1.047661e+06,0.001676,0.009839


In [27]:
# ---------------------------------------------------------------------------
# Cell 22 — Final validation of GeoCANOE CO2 storage integration products
# ---------------------------------------------------------------------------

# Validate the two products intended for promotion to:
#
#     data_files/processed/co2_storage/
#
# Products:
#   1. regional_storage_evidence
#   2. storage_region_crosswalk
#
# This cell performs reconciliation and semantic QA only.
# It does not modify either product.


# ---------------------------------------------------------------------------
# 1. Structural QA
# ---------------------------------------------------------------------------

assert len(regional_storage_evidence) == len(regions_25km)
assert regional_storage_evidence["region"].is_unique

assert len(storage_region_crosswalk) == len(storage_region_intersections)

assert not storage_region_crosswalk.duplicated(
    ["region", "site_id", "storage_feature_id"]
).any()

assert regional_storage_evidence.crs == regions_25km.crs
assert storage_region_crosswalk.crs == regions_25km.crs


# ---------------------------------------------------------------------------
# 2. Referential integrity
# ---------------------------------------------------------------------------

assert storage_region_crosswalk["region"].isin(
    regional_storage_evidence["region"]
).all()

assert storage_region_crosswalk["storage_feature_id"].isin(
    storage_features_projected["storage_feature_id"]
).all()

assert storage_region_crosswalk["storage_unit_id"].isin(
    storage_units["storage_unit_id"]
).all()


# ---------------------------------------------------------------------------
# 3. Spatial-domain QA
# ---------------------------------------------------------------------------

assert (
    storage_region_crosswalk["intersection_area_m2"] >= 0
).all()

assert (
    storage_region_crosswalk["region_overlap_fraction"]
    .between(0, 1)
    .all()
)

assert (
    storage_region_crosswalk["feature_overlap_fraction"]
    .between(0, 1)
    .all()
)

assert (
    regional_storage_evidence[
        "capacity_evidence_coverage_fraction"
    ]
    .between(0, 1)
    .all()
)


# ---------------------------------------------------------------------------
# 4. Regional evidence consistency
# ---------------------------------------------------------------------------

# Every region appearing in the crosswalk must be geologically accessible.

crosswalk_regions = set(
    storage_region_crosswalk["region"].unique()
)

accessible_regions = set(
    regional_storage_evidence.loc[
        regional_storage_evidence["storage_accessible"],
        "region",
    ]
)

assert crosswalk_regions == accessible_regions


# Capacity evidence must be a subset of quantitative geological evidence.

assert (
    regional_storage_evidence.loc[
        regional_storage_evidence["has_any_capacity_evidence"],
        "has_quantitative_storage_evidence",
    ]
).all()


# Positive capacity evidence should correspond to positive unioned coverage.

assert (
    regional_storage_evidence[
        "has_any_capacity_evidence"
    ]
    ==
    (
        regional_storage_evidence[
            "capacity_evidence_coverage_fraction"
        ] > 0
    )
).all()


# ---------------------------------------------------------------------------
# 5. Source-presence reconciliation
# ---------------------------------------------------------------------------

source_flag_map = {
    "NATCARB": "has_natcarb",
    "BC_STORAGE_ATLAS": "has_bc_storage_atlas",
    "ATLANTIC_COS": "has_atlantic_cos",
}

source_validation = []

for source, flag in source_flag_map.items():

    crosswalk_count = (
        storage_region_crosswalk.loc[
            storage_region_crosswalk["source_dataset"] == source,
            "region",
        ]
        .nunique()
    )

    regional_count = int(
        regional_storage_evidence[flag].sum()
    )

    source_validation.append(
        {
            "source_dataset": source,
            "crosswalk_regions": crosswalk_count,
            "regional_evidence_regions": regional_count,
            "match": crosswalk_count == regional_count,
        }
    )

source_validation = pd.DataFrame(source_validation)

assert source_validation["match"].all()


# ---------------------------------------------------------------------------
# 6. Final reconciliation summary
# ---------------------------------------------------------------------------

validation_summary = pd.DataFrame(
    [
        {
            "metric": "GeoCANOE basemap regions",
            "count": len(regions_25km),
        },
        {
            "metric": "Regions with geological storage evidence",
            "count": regional_storage_evidence[
                "storage_accessible"
            ].sum(),
        },
        {
            "metric": "Regions with quantitative geological evidence",
            "count": regional_storage_evidence[
                "has_quantitative_storage_evidence"
            ].sum(),
        },
        {
            "metric": "Regions with qualitative geological evidence",
            "count": regional_storage_evidence[
                "has_qualitative_storage_evidence"
            ].sum(),
        },
        {
            "metric": "Regions with positive capacity evidence",
            "count": regional_storage_evidence[
                "has_any_capacity_evidence"
            ].sum(),
        },
        {
            "metric": "Storage-region crosswalk records",
            "count": len(storage_region_crosswalk),
        },
        {
            "metric": "Storage features represented",
            "count": storage_region_crosswalk[
                "storage_feature_id"
            ].nunique(),
        },
        {
            "metric": "Logical storage units represented",
            "count": storage_region_crosswalk[
                "storage_unit_id"
            ].nunique(),
        },
    ]
)


print("FINAL CO2 STORAGE INTEGRATION VALIDATION")
print("=" * 80)

print("\nSOURCE RECONCILIATION")
display(source_validation)

print("\nPRODUCT RECONCILIATION")
display(validation_summary)

print("\nQA STATUS")
print("PASS — regional_storage_evidence")
print("PASS — storage_region_crosswalk")
print("PASS — source → region reconciliation")
print("PASS — spatial-domain constraints")
print("PASS — referential integrity")

FINAL CO2 STORAGE INTEGRATION VALIDATION

SOURCE RECONCILIATION


,source_dataset,crosswalk_regions,regional_evidence_regions,match
0,NATCARB,2579,2579,True
1,BC_STORAGE_ATLAS,157,157,True
2,ATLANTIC_COS,156,156,True



PRODUCT RECONCILIATION


,metric,count
0,GeoCANOE basemap regions,9269
1,Regions with geological storage evidence,2735
2,Regions with quantitative geological evidence,2579
3,Regions with qualitative geological evidence,156
4,Regions with positive capacity evidence,1483
5,Storage-region crosswalk records,61748
6,Storage features represented,30637
7,Logical storage units represented,2819



QA STATUS
PASS — regional_storage_evidence
PASS — storage_region_crosswalk
PASS — source → region reconciliation
PASS — spatial-domain constraints
PASS — referential integrity


## Notebook 17 — Integration Checkpoint

The exploratory geological CO₂ storage integration workflow is now validated for the
25 km `provinces_only` GeoCANOE basemap.

### Validated processed products

Two complementary GeoCANOE spatial products have been defined:

1. **`regional_storage_evidence`**
   - one record per GeoCANOE region;
   - preserves source-specific geological evidence;
   - distinguishes quantitative and qualitative evidence;
   - identifies available capacity-assessment classes;
   - stores unioned positive-capacity coverage as a continuous spatial metric.

2. **`storage_region_crosswalk`**
   - preserves the many-to-many relationship between storage features and GeoCANOE regions;
   - retains storage feature and logical storage-unit identifiers;
   - preserves source, storage type, representation, and assessment metadata;
   - records intersection area and bidirectional overlap fractions;
   - does not allocate or aggregate geological storage capacity.

### Intended architecture

The unified CanCO₂ storage database and GeoCANOE basemap are both treated as
upstream Silver products. Their spatial integration produces a GeoCANOE-specific
processed Silver layer:

CanCO₂ Silver + GeoCANOE Basemap Silver
→ GeoCANOE CO₂ Storage Integration
→ `data_files/processed/co2_storage/`
→ Gold SQL / model parameter construction
→ Temoa / Pyomo

The processed CO₂ storage layer therefore acts as the explicit spatial handshake
between external geological storage information and GeoCANOE's regional ontology.

### Modeling boundary

No finite geological storage capacity is assigned to GeoCANOE regions in this
notebook.

A storage resource may intersect multiple model regions, and multiple source
datasets may describe overlapping or potentially identical geological resources.
Consequently, directly summing source capacities by GeoCANOE region would risk
spatial duplication and cross-source double counting.

The current processed layer instead preserves topology and evidence. Finite
capacity representation, storage-unit reconciliation, capacity scenarios
(P10/P50/P90, theoretical, effective), and Gold SQL constraints are deferred to
subsequent development.

### Configuration principle

Continuous spatial evidence metrics are persisted in the processed layer.
Scenario assumptions are applied downstream.

For example:

`capacity_evidence_coverage_fraction`

is persisted, while a scenario-specific rule such as:

`minimum_storage_coverage = 0.01`

should be applied during Gold/model construction rather than embedded permanently
in the spatial preprocessing product.